# 01. Prüfen und die Güte messen

In der 2. Runde gibt es oft keine Lösung, von der man weiß, dass sie die beste ist. Du baust ein Verfahren, das **irgendeine** brauchbare Lösung liefert
(eine Heuristik, siehe [Notebook 16](https://ktheu.github.io/bwinfN/notebooks/index.html?path=learn_bwinf1/16_heuristiken.ipynb) von learn_bwinf1). Dann stellen sich zwei Fragen, die in der Dokumentation über Punkte entscheiden:

1. **Ist die Lösung überhaupt erlaubt?** Ein Programm, das schöne Zahlen ausgibt, aber eine Regel verletzt, ist wertlos.
2. **Wie gut ist die Lösung?** In Notebook 16 stand nur „gib die Qualität an“. Hier lernst du, **wie** man das macht, auch wenn man das Optimum nicht kennt.

In diesem Notebook lernst du:

- einen **Schiedsrichter** zu schreiben: ein eigenes, einfaches Programm, das jede Lösung nachprüft und den **ersten Verstoß genau** meldet,
- **Funktionen wie Werte zu übergeben** (neu in learn_bwinf2) und so mehrere Verfahren mit demselben Schiedsrichter zu vergleichen,
- bei kleinen Fällen die **exakte Lösung** zum Vergleich zu berechnen (Durchprobieren wie in [Notebook 1](https://ktheu.github.io/bwinfN/notebooks/index.html?path=learn_bwinf1/01_durchprobieren.ipynb), mit Abschneiden wie in [Notebook 3](https://ktheu.github.io/bwinfN/notebooks/index.html?path=learn_bwinf1/03_schranken_symmetrien.ipynb)),
- **untere Schranken** herzuleiten und zu begründen. Sie dienen hier nicht zum Abschneiden, sondern zum **Bewerten**,
- die **Güte** als Abstand zur Schranke in Prozent über viele Zufallsfälle als Tabelle zu messen und **ehrlich** zu formulieren, was man daraus folgern darf,
- zu sehen, wie ein Solver (MiniZinc) die Güte auch für mittelgroße Fälle messbar macht, und wo er an Grenzen kommt.

Das neutrale Beispiel sind **Pakete, die auf Lieferwagen verteilt werden**. In den Wettbewerbs-Ecken prüfst du Pläne für eine Drohnenlieferung (nach Aufgabe „Missglückte Drohnenlieferung“)
und baust eine Mauer ohne übereinanderliegende Fugen (nach Aufgabe „Die Kunst der Fuge“).

## 1. Der Schiedsrichter

Unser Beispiel: Ein Paketdienst hat 9 Pakete und Lieferwagen, die **höchstens 20 kg** tragen. Gesucht ist eine Verteilung mit **möglichst wenigen Wagen**
(das **Behälterproblem**, englisch *bin packing*).

```python
K = 20                                       # Kapazität eines Wagens in kg
pakete = [7, 9, 10, 7, 4, 7, 8, 3, 4]        # Gewichte der Pakete in kg (Paket 0 wiegt 7 kg, Paket 1 wiegt 9 kg, ...)
```

Eine **Lösung** ist eine Liste `zuordnung`: `zuordnung[i]` ist die Nummer des Wagens, auf den Paket `i` kommt. `[0, 0, 1, 1, 2, 2, 3, 3, 3]` heißt:
Paket 0 und 1 auf Wagen 0, Paket 2 und 3 auf Wagen 1 und so weiter, insgesamt 4 Wagen.

Bevor wir eine Lösung bewerten, brauchen wir den **Schiedsrichter**: eine Funktion, die ohne jede Schlauheit nachrechnet, ob **alle Regeln** eingehalten sind. Sie ist unabhängig
vom Verfahren, das die Lösung gefunden hat. Eine **gute Meldung** nennt den **ersten Verstoß genau** (welcher Wagen, wie schwer), denn damit findest du den Fehler im Verfahren schnell.

#### Aufgabe 1.1

Schreibe `pruefe(pakete, K, zuordnung)`. Die Funktion gibt `'ok'` zurück, wenn die Lösung erlaubt ist, sonst einen Satz, der den **ersten** Verstoß beschreibt. Drei Regeln gelten:

- Die Liste `zuordnung` hat **einen Eintrag je Paket**.
- Wagennummern sind **nicht negativ**.
- Kein Wagen trägt **mehr als `K`** kg. Prüfe die Wagen in der Reihenfolge ihrer Nummern.

Dazu sollst du `wagenzahl(zuordnung)` schreiben: die Zahl der **verschiedenen** Wagen, die benutzt werden (`set(liste)` entfernt Doppelte).

**Erwartete Ausgabe:**
```
ok
Wagen 0 ist überladen: 26 kg bei höchstens 20 kg
3 Einträge für 9 Pakete
Paket 8 hat die ungültige Wagennummer -1
```

In [ ]:
K = 20                                       # Kapazität eines Wagens in kg
pakete = [7, 9, 10, 7, 4, 7, 8, 3, 4]        # Gewichte der Pakete in kg

def pruefe(pakete, K, zuordnung):
    """Gibt 'ok' zurück oder beschreibt den ersten Verstoß."""
    # dein Code
    return 'ok'


def wagenzahl(zuordnung):
    return 0

print(pruefe(pakete, K, [0, 0, 1, 1, 2, 2, 3, 3, 3]))
print(pruefe(pakete, K, [0, 0, 0, 1, 1, 1, 2, 2, 2]))
print(pruefe(pakete, K, [0, 1, 2]))
print(pruefe(pakete, K, [0, 0, 1, 1, 2, 2, 3, 3, -1]))

<details>
<summary>Tipp anzeigen</summary>

Erst die Länge prüfen. Dann ein Dictionary `last`, das zu jedem benutzten Wagen das Gewicht summiert (`last[w] = last.get(w, 0) + pakete[i]`). Zum Schluss `for w in sorted(last):`
und bei `last[w] > K` den Satz zurückgeben. `return 'ok'` steht ganz am Ende, erst wenn alle Prüfungen bestanden sind.

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
K = 20                                       # Kapazität eines Wagens in kg
pakete = [7, 9, 10, 7, 4, 7, 8, 3, 4]        # Gewichte der Pakete in kg

def pruefe(pakete, K, zuordnung):
    """Gibt 'ok' zurück oder beschreibt den ersten Verstoß."""
    if len(zuordnung) != len(pakete):
        return f'{len(zuordnung)} Einträge für {len(pakete)} Pakete'
    last = {}                                # last[w]: Gewicht auf Wagen w
    for i in range(len(pakete)):
        w = zuordnung[i]
        if w < 0:
            return f'Paket {i} hat die ungültige Wagennummer {w}'
        last[w] = last.get(w, 0) + pakete[i]
    for w in sorted(last):
        if last[w] > K:
            return f'Wagen {w} ist überladen: {last[w]} kg bei höchstens {K} kg'
    return 'ok'


def wagenzahl(zuordnung):
    return len(set(zuordnung))

print(pruefe(pakete, K, [0, 0, 1, 1, 2, 2, 3, 3, 3]))
print(pruefe(pakete, K, [0, 0, 0, 1, 1, 1, 2, 2, 2]))
print(pruefe(pakete, K, [0, 1, 2]))
print(pruefe(pakete, K, [0, 0, 1, 1, 2, 2, 3, 3, -1]))
```

Die erste Verteilung ist erlaubt (16, 17, 11 und 15 kg). In der zweiten trägt Wagen 0 genau 7 + 9 + 10 = 26 kg. Die dritte hat zu wenige Einträge. Die vierte hat eine negative Nummer.
Wichtig ist die **Reihenfolge der Prüfungen**: Erst prüft man, ob die Eingabe überhaupt die richtige Form hat, dann die eigentlichen Regeln.

</details>

Der Schiedsrichter ist **kein Teil des Verfahrens**. Er wird für **jede** Lösung aufgerufen, auch für die, die du von Hand eingibst oder aus einer Datei einliest.

#### Frage 1.2

1. Warum soll der Schiedsrichter **ohne Hilfe** aus dem Lösungsverfahren geschrieben werden (er soll zum Beispiel keine Funktion des Verfahrens benutzen)?
2. In der 2. Runde schreibst du die Lösung in eine **Datei**. Was solltest du am Ende **zusätzlich** tun, damit der Schiedsrichter wirklich die abgegebene Lösung prüft?

✏️ *Deine Antwort (Doppelklick zum Bearbeiten):*

<details>
<summary>Lösung anzeigen (erst selbst überlegen!)</summary>

1. Macht das Verfahren einen Denkfehler, macht ihn der Schiedsrichter sonst **mit** und meldet „ok“. Ein unabhängiger, **dümmerer** Schiedsrichter rechnet die Regeln aus der Aufgabenstellung nach, ohne
   auf Tricks des Verfahrens zu vertrauen. Er ist kurz genug, dass man ihn leicht überblickt.
2. Die Datei **neu einlesen** und das Ergebnis des Einlesens prüfen. So findest du auch Fehler beim Schreiben (falsches Format, vertauschte Spalten, abgeschnittene Zeilen).
   Dass die Liste im Programm stimmt, hilft dem Leser der Datei nicht.

</details>

### Finde den Fehler: ein zu milder Schiedsrichter

Jemand hat einen Schiedsrichter geschrieben, der nur prüft, ob **insgesamt** genug Platz da ist. Er meldet bei der überladenen Verteilung von eben trotzdem „ok“:

In [ ]:
K = 20                                       # Kapazität eines Wagens in kg
pakete = [7, 9, 10, 7, 4, 7, 8, 3, 4]        # Gewichte der Pakete in kg

def pruefe_mild(pakete, K, zuordnung):
    if len(zuordnung) != len(pakete):
        return 'falsche Länge'
    if sum(pakete) > K * len(set(zuordnung)):      # insgesamt zu schwer für die benutzten Wagen?
        return 'überladen'
    return 'ok'


print(pruefe_mild(pakete, K, [0, 0, 0, 1, 1, 1, 2, 2, 2]))
print(pruefe(pakete, K, [0, 0, 0, 1, 1, 1, 2, 2, 2]))

ok
Wagen 0 ist überladen: 26 kg bei höchstens 20 kg


#### Frage 1.3

Die Gesamtmasse der Pakete ist 59 kg, drei Wagen tragen zusammen 60 kg. Erkläre, warum `pruefe_mild` trotzdem falsch ist, und nenne eine Eingabe, bei der er „überladen“ meldet,
obwohl die Lösung erlaubt ist (oder erkläre, warum es die nicht gibt). Was ist schlimmer, ein zu strenger oder ein zu milder Schiedsrichter?

✏️ *Deine Antwort (Doppelklick zum Bearbeiten):*

<details>
<summary>Lösung anzeigen (erst selbst überlegen!)</summary>

Die Regel gilt **für jeden einzelnen Wagen**, nicht für die Summe. Mit 26 kg auf Wagen 0 (und 18 und 15 kg auf den anderen) ist die Summe 59 kg nicht größer als 60 kg, die Verteilung aber verboten.
Wenn die Summe zu groß ist, ist mindestens ein Wagen überladen, das heißt, „überladen“ meldet `pruefe_mild` nur zu Recht. Er ist also **nie zu streng, nur zu mild**.
Ein zu **milder** Schiedsrichter ist schlimmer: Er lässt falsche Lösungen als „ok“ durch, und du gibst sie ab, ohne es zu merken. Ein zu strenger meldet dir Fehler, die keine sind, das fällt wenigstens auf.

</details>

## 2. Zwei einfache Verfahren, Funktionen als Werte

**First Fit** (erster passender Wagen): Man nimmt die Pakete **der Reihe nach**. Jedes Paket kommt auf den **ersten** Wagen, auf dem noch Platz ist. Gibt es keinen, nimmt man einen neuen Wagen.

Eine Verbesserung ist **First Fit Decreasing** (nach Größe absteigend): Man sortiert die Pakete vorher und beginnt mit den **schwersten**. Das ist ein gieriges Verfahren wie in
[Notebook 12](https://ktheu.github.io/bwinfN/notebooks/index.html?path=learn_bwinf1/12_greedy.ipynb), und man prüft es auch wie dort durch Vergleich mit Gegenbeispielen.

#### Aufgabe 2.1

Schreibe `first_fit(pakete, K, reihenfolge)`. Der Parameter `reihenfolge` ist eine Liste der Paketnummern. Die Pakete werden in **dieser Reihenfolge** verteilt, `zuordnung[i]` bleibt aber
immer die Nummer des Wagens für **Paket `i`**. Ein Wagen merkt sich seine **freie** Kapazität. Pakete, die schon passen, kommen auf den **ersten** passenden Wagen.

**Erwartete Ausgabe:**
```
[0, 0, 1, 1, 0, 2, 2, 1, 2]
3 ok
```

In [ ]:
K = 20                                       # Kapazität eines Wagens in kg
pakete = [7, 9, 10, 7, 4, 7, 8, 3, 4]        # Gewichte der Pakete in kg

def pruefe(pakete, K, zuordnung):
    """Gibt 'ok' zurück oder beschreibt den ersten Verstoß."""
    if len(zuordnung) != len(pakete):
        return f'{len(zuordnung)} Einträge für {len(pakete)} Pakete'
    last = {}                                # last[w]: Gewicht auf Wagen w
    for i in range(len(pakete)):
        w = zuordnung[i]
        if w < 0:
            return f'Paket {i} hat die ungültige Wagennummer {w}'
        last[w] = last.get(w, 0) + pakete[i]
    for w in sorted(last):
        if last[w] > K:
            return f'Wagen {w} ist überladen: {last[w]} kg bei höchstens {K} kg'
    return 'ok'


def wagenzahl(zuordnung):
    return len(set(zuordnung))

def first_fit(pakete, K, reihenfolge):
    frei = []                                # frei[w]: noch freie Kapazität von Wagen w
    zuordnung = [0] * len(pakete)
    for i in reihenfolge:
        # dein Code: den ersten Wagen mit frei[w] >= pakete[i] suchen, sonst einen neuen anfangen
        pass
    return zuordnung

zuordnung = first_fit(pakete, K, list(range(len(pakete))))
print(zuordnung)
print(wagenzahl(zuordnung), pruefe(pakete, K, zuordnung))

<details>
<summary>Tipp anzeigen</summary>

`gefunden = -1`, dann `for w in range(len(frei)):` und beim ersten passenden Wagen `gefunden = w` und `break`. War `gefunden` danach noch -1, gibt es keinen:
`frei.append(K)` und `gefunden = len(frei) - 1`. Dann `frei[gefunden]` verkleinern und `zuordnung[i] = gefunden` setzen.

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
K = 20                                       # Kapazität eines Wagens in kg
pakete = [7, 9, 10, 7, 4, 7, 8, 3, 4]        # Gewichte der Pakete in kg

def pruefe(pakete, K, zuordnung):
    """Gibt 'ok' zurück oder beschreibt den ersten Verstoß."""
    if len(zuordnung) != len(pakete):
        return f'{len(zuordnung)} Einträge für {len(pakete)} Pakete'
    last = {}                                # last[w]: Gewicht auf Wagen w
    for i in range(len(pakete)):
        w = zuordnung[i]
        if w < 0:
            return f'Paket {i} hat die ungültige Wagennummer {w}'
        last[w] = last.get(w, 0) + pakete[i]
    for w in sorted(last):
        if last[w] > K:
            return f'Wagen {w} ist überladen: {last[w]} kg bei höchstens {K} kg'
    return 'ok'


def wagenzahl(zuordnung):
    return len(set(zuordnung))

def first_fit(pakete, K, reihenfolge):
    """reihenfolge: Liste der Paketnummern in der Reihenfolge, in der sie verteilt werden."""
    frei = []                                # frei[w]: noch freie Kapazität von Wagen w
    zuordnung = [0] * len(pakete)
    for i in reihenfolge:
        gefunden = -1
        for w in range(len(frei)):
            if frei[w] >= pakete[i]:
                gefunden = w
                break                        # der erste passende Wagen
        if gefunden == -1:                   # keiner passt: neuer Wagen
            frei.append(K)
            gefunden = len(frei) - 1
        frei[gefunden] = frei[gefunden] - pakete[i]
        zuordnung[i] = gefunden
    return zuordnung

zuordnung = first_fit(pakete, K, list(range(len(pakete))))
print(zuordnung)
print(wagenzahl(zuordnung), pruefe(pakete, K, zuordnung))
```

Die Pakete 0, 1 und 4 (7 + 9 + 4 = 20 kg) kommen auf Wagen 0, die Pakete 2, 3 und 7 (10 + 7 + 3 = 20 kg) auf Wagen 1 und die Pakete 5, 6 und 8 (7 + 8 + 4 = 19 kg) auf Wagen 2. Der Schiedsrichter sagt „ok“, die Lösung hat 3 Wagen.

</details>

Beide Verfahren sehen jetzt gleich aus: Sie bekommen `(pakete, K)` und geben eine `zuordnung` zurück. Das nutzen wir gleich.
`ff_gegeben` nimmt die Pakete in der **gegebenen** Reihenfolge, `ffd` sortiert zuerst. Das `key=lambda i: -pakete[i]` sortiert die Paketnummern nach **negativem** Gewicht, also das schwerste Paket zuerst.

In [ ]:
K = 20                                       # Kapazität eines Wagens in kg
pakete = [7, 9, 10, 7, 4, 7, 8, 3, 4]        # Gewichte der Pakete in kg

def pruefe(pakete, K, zuordnung):
    """Gibt 'ok' zurück oder beschreibt den ersten Verstoß."""
    if len(zuordnung) != len(pakete):
        return f'{len(zuordnung)} Einträge für {len(pakete)} Pakete'
    last = {}                                # last[w]: Gewicht auf Wagen w
    for i in range(len(pakete)):
        w = zuordnung[i]
        if w < 0:
            return f'Paket {i} hat die ungültige Wagennummer {w}'
        last[w] = last.get(w, 0) + pakete[i]
    for w in sorted(last):
        if last[w] > K:
            return f'Wagen {w} ist überladen: {last[w]} kg bei höchstens {K} kg'
    return 'ok'


def wagenzahl(zuordnung):
    return len(set(zuordnung))

def first_fit(pakete, K, reihenfolge):
    """reihenfolge: Liste der Paketnummern in der Reihenfolge, in der sie verteilt werden."""
    frei = []                                # frei[w]: noch freie Kapazität von Wagen w
    zuordnung = [0] * len(pakete)
    for i in reihenfolge:
        gefunden = -1
        for w in range(len(frei)):
            if frei[w] >= pakete[i]:
                gefunden = w
                break                        # der erste passende Wagen
        if gefunden == -1:                   # keiner passt: neuer Wagen
            frei.append(K)
            gefunden = len(frei) - 1
        frei[gefunden] = frei[gefunden] - pakete[i]
        zuordnung[i] = gefunden
    return zuordnung

def ff_gegeben(pakete, K):
    return first_fit(pakete, K, list(range(len(pakete))))


def ffd(pakete, K):
    # Paketnummern nach Gewicht absteigend: key=lambda wie in Notebook 03 von learn_bwinf1
    reihenfolge = sorted(range(len(pakete)), key=lambda i: -pakete[i])
    return first_fit(pakete, K, reihenfolge)

### Funktionen kann man wie Werte übergeben

In learn_bwinf2 darfst du, anders als in learn_bwinf1, **Funktionen als Argumente** übergeben. Der Name einer Funktion ohne Klammern (`verdopple`) ist ein Wert wie eine Zahl oder eine Liste.
Man kann ihn in einer Variablen speichern, in eine Liste legen und an andere Funktionen weitergeben. **Erst mit Klammern (`f(3)`) wird die Funktion ausgeführt.**

#### Aufgabe 2.2: Was wird ausgegeben?

Überlege zuerst, was der Code ausgibt, und schreibe es auf. Führe die Zelle erst danach aus.

In [ ]:
def verdopple(x):
    return 2 * x


def quadrat(x):
    return x * x


def wende_an(funktion, wert):
    return funktion(wert)


print(wende_an(verdopple, 5))
print(wende_an(quadrat, 5))
liste = [verdopple, quadrat]
for f in liste:
    print(f(3))

<details>
<summary>Lösung anzeigen (erst selbst überlegen!)</summary>

**Ausgabe:**
```
10
25
6
9
```

`wende_an(verdopple, 5)` ruft innen `verdopple(5)` auf, das ist 10, und `wende_an(quadrat, 5)` gibt 25. In der Schleife läuft `f` erst über `verdopple` (6), dann über `quadrat` (9).
Der Parameter heißt `funktion` und kann jede Funktion mit **einem** Parameter enthalten. Genau so übergeben wir gleich unsere Verfahren.

</details>

#### Aufgabe 2.3

Schreibe `bewerte(verfahren, pakete, K)`: Sie ruft das **übergebene** Verfahren mit `(pakete, K)` auf, lässt die Lösung vom Schiedsrichter prüfen und gibt die Zahl der Wagen zurück.
Meldet der Schiedsrichter etwas anderes als `'ok'`, soll `bewerte` einen Fehler auslösen: `raise ValueError('Schiedsrichter: ' + urteil)`.
So kann kein Verfahren eine ungeprüfte Lösung in eine Tabelle schmuggeln.

**Erwartete Ausgabe:**
```
First Fit 3
First Fit Decreasing 4
```

In [ ]:
K = 20                                       # Kapazität eines Wagens in kg
pakete = [7, 9, 10, 7, 4, 7, 8, 3, 4]        # Gewichte der Pakete in kg

def pruefe(pakete, K, zuordnung):
    """Gibt 'ok' zurück oder beschreibt den ersten Verstoß."""
    if len(zuordnung) != len(pakete):
        return f'{len(zuordnung)} Einträge für {len(pakete)} Pakete'
    last = {}                                # last[w]: Gewicht auf Wagen w
    for i in range(len(pakete)):
        w = zuordnung[i]
        if w < 0:
            return f'Paket {i} hat die ungültige Wagennummer {w}'
        last[w] = last.get(w, 0) + pakete[i]
    for w in sorted(last):
        if last[w] > K:
            return f'Wagen {w} ist überladen: {last[w]} kg bei höchstens {K} kg'
    return 'ok'


def wagenzahl(zuordnung):
    return len(set(zuordnung))

def first_fit(pakete, K, reihenfolge):
    """reihenfolge: Liste der Paketnummern in der Reihenfolge, in der sie verteilt werden."""
    frei = []                                # frei[w]: noch freie Kapazität von Wagen w
    zuordnung = [0] * len(pakete)
    for i in reihenfolge:
        gefunden = -1
        for w in range(len(frei)):
            if frei[w] >= pakete[i]:
                gefunden = w
                break                        # der erste passende Wagen
        if gefunden == -1:                   # keiner passt: neuer Wagen
            frei.append(K)
            gefunden = len(frei) - 1
        frei[gefunden] = frei[gefunden] - pakete[i]
        zuordnung[i] = gefunden
    return zuordnung

def ff_gegeben(pakete, K):
    return first_fit(pakete, K, list(range(len(pakete))))


def ffd(pakete, K):
    # Paketnummern nach Gewicht absteigend: key=lambda wie in Notebook 03 von learn_bwinf1
    reihenfolge = sorted(range(len(pakete)), key=lambda i: -pakete[i])
    return first_fit(pakete, K, reihenfolge)

def bewerte(verfahren, pakete, K):
    # dein Code
    return 0


for name, verfahren in [('First Fit', ff_gegeben), ('First Fit Decreasing', ffd)]:
    print(name, bewerte(verfahren, pakete, K))

<details>
<summary>Tipp anzeigen</summary>

`zuordnung = verfahren(pakete, K)`, dann `urteil = pruefe(pakete, K, zuordnung)`. Wenn `urteil != 'ok'`: `raise ValueError(...)`. Sonst `return wagenzahl(zuordnung)`.

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
K = 20                                       # Kapazität eines Wagens in kg
pakete = [7, 9, 10, 7, 4, 7, 8, 3, 4]        # Gewichte der Pakete in kg

def pruefe(pakete, K, zuordnung):
    """Gibt 'ok' zurück oder beschreibt den ersten Verstoß."""
    if len(zuordnung) != len(pakete):
        return f'{len(zuordnung)} Einträge für {len(pakete)} Pakete'
    last = {}                                # last[w]: Gewicht auf Wagen w
    for i in range(len(pakete)):
        w = zuordnung[i]
        if w < 0:
            return f'Paket {i} hat die ungültige Wagennummer {w}'
        last[w] = last.get(w, 0) + pakete[i]
    for w in sorted(last):
        if last[w] > K:
            return f'Wagen {w} ist überladen: {last[w]} kg bei höchstens {K} kg'
    return 'ok'


def wagenzahl(zuordnung):
    return len(set(zuordnung))

def first_fit(pakete, K, reihenfolge):
    """reihenfolge: Liste der Paketnummern in der Reihenfolge, in der sie verteilt werden."""
    frei = []                                # frei[w]: noch freie Kapazität von Wagen w
    zuordnung = [0] * len(pakete)
    for i in reihenfolge:
        gefunden = -1
        for w in range(len(frei)):
            if frei[w] >= pakete[i]:
                gefunden = w
                break                        # der erste passende Wagen
        if gefunden == -1:                   # keiner passt: neuer Wagen
            frei.append(K)
            gefunden = len(frei) - 1
        frei[gefunden] = frei[gefunden] - pakete[i]
        zuordnung[i] = gefunden
    return zuordnung

def ff_gegeben(pakete, K):
    return first_fit(pakete, K, list(range(len(pakete))))


def ffd(pakete, K):
    # Paketnummern nach Gewicht absteigend: key=lambda wie in Notebook 03 von learn_bwinf1
    reihenfolge = sorted(range(len(pakete)), key=lambda i: -pakete[i])
    return first_fit(pakete, K, reihenfolge)

def bewerte(verfahren, pakete, K):
    """Führt verfahren(pakete, K) aus, lässt den Schiedsrichter prüfen und gibt die Zahl der Wagen zurück."""
    zuordnung = verfahren(pakete, K)
    urteil = pruefe(pakete, K, zuordnung)
    if urteil != 'ok':
        raise ValueError('Schiedsrichter: ' + urteil)
    return wagenzahl(zuordnung)

for name, verfahren in [('First Fit', ff_gegeben), ('First Fit Decreasing', ffd)]:
    print(name, bewerte(verfahren, pakete, K))
```

Hier liefert die **sortierte** Variante mit 4 Wagen ein **schlechteres** Ergebnis als die unsortierte mit 3. Das widerspricht der Faustregel „erst die großen“.

</details>

#### Frage 2.4

1. Folgt aus diesem einen Beispiel, dass First Fit **besser** ist als First Fit Decreasing? Was folgt wirklich?
2. Wie viele Wagen braucht man hier **mindestens**? Überlege mit der Gesamtmasse (59 kg).

✏️ *Deine Antwort (Doppelklick zum Bearbeiten):*

<details>
<summary>Lösung anzeigen (erst selbst überlegen!)</summary>

1. Nein. Ein einzelnes Beispiel zeigt nur: First Fit Decreasing ist **nicht immer** besser. Für eine Aussage über „meistens“ braucht man **viele** Fälle (Abschnitt 5).
2. 59 kg passen nicht auf zwei Wagen (40 kg), also braucht man **mindestens 3**. First Fit schafft 3, ist damit also **optimal**. First Fit Decreasing schafft nur 4, das ist **nicht** optimal.
   Genau diese Überlegung („Wie gut kann es höchstens gehen?“) ist eine **Schranke** (Abschnitt 4).

</details>

## 3. Das Optimum für kleine Fälle: Durchprobieren

Wie weit sind die Verfahren vom **besten** Ergebnis entfernt? Für **kleine** Fälle kann man es ausrechnen, indem man alle Verteilungen durchprobiert ([Notebook 1](https://ktheu.github.io/bwinfN/notebooks/index.html?path=learn_bwinf1/01_durchprobieren.ipynb)).
Jedes Paket kommt auf einen schon benutzten Wagen mit genug Platz oder auf einen **neuen**. Weil die Wagen gleich sind, brauchen wir sie nicht durchzunummerieren: Ein neuer Wagen
ist immer der nächste, so zählen wir die Verteilungen nicht mehrfach (wie bei der Symmetrie in [Notebook 3](https://ktheu.github.io/bwinfN/notebooks/index.html?path=learn_bwinf1/03_schranken_symmetrien.ipynb)).
Die Zahl der Möglichkeiten sind die **Bell-Zahlen**: Für 9 Pakete etwa 21 147, für 12 Pakete etwa 4,2 Millionen und für 15 Pakete schon über 1,3 Milliarden.

Und wir **schneiden ab**, wie bei Branch and Bound: Sind schon so viele Wagen offen wie in der besten bisher gefundenen Lösung, kann der Rest nicht mehr besser werden.

#### Aufgabe 3.1

Ergänze `exakt(pakete, K)`: Die Funktion `probiere(i)` verteilt Paket `i` und alle folgenden. Probiere für Paket `i` **jeden** schon offenen Wagen, auf dem noch Platz ist (Paket drauflegen,
weitermachen, **wieder wegnehmen**), und danach einen **neuen** Wagen (anhängen, weitermachen, wieder entfernen). Die Funktion gibt die **kleinste Zahl der Wagen** zurück
und, wie oft `probiere` aufgerufen wurde.

**Erwartete Ausgabe:**
```
(3, 30)
(2, 7)
```

In [ ]:
K = 20                                       # Kapazität eines Wagens in kg
pakete = [7, 9, 10, 7, 4, 7, 8, 3, 4]        # Gewichte der Pakete in kg

def exakt(pakete, K):
    n = len(pakete)
    frei = []                                # frei[w]: noch freie Kapazität von Wagen w
    stand = {'beste': n + 1, 'aufrufe': 0}   # ein Dictionary, damit die innere Funktion die Werte ändern kann

    def probiere(i):
        stand['aufrufe'] = stand['aufrufe'] + 1
        if len(frei) >= stand['beste']:      # nicht besser als die beste Lösung: abschneiden
            return
        if i == n:                           # alle Pakete verteilt, und weniger Wagen als bisher
            stand['beste'] = len(frei)
            return
        # dein Code: vorhandene Wagen probieren, dann einen neuen Wagen

    probiere(0)
    return stand['beste'], stand['aufrufe']


print(exakt(pakete, K))
print(exakt([4, 4, 4, 4], 8))

<details>
<summary>Tipp anzeigen</summary>

`for w in range(len(frei)):` Wenn `frei[w] >= pakete[i]`: `frei[w] = frei[w] - pakete[i]`, `probiere(i + 1)`, `frei[w] = frei[w] + pakete[i]`.
Danach `frei.append(K - pakete[i])`, `probiere(i + 1)`, `frei.pop()`.

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
K = 20                                       # Kapazität eines Wagens in kg
pakete = [7, 9, 10, 7, 4, 7, 8, 3, 4]        # Gewichte der Pakete in kg

def exakt(pakete, K):
    """Kleinste mögliche Zahl der Wagen und Zahl der Aufrufe."""
    n = len(pakete)
    frei = []                                # frei[w]: noch freie Kapazität von Wagen w
    stand = {'beste': n + 1, 'aufrufe': 0}   # ein Dictionary, damit die innere Funktion die Werte ändern kann

    def probiere(i):
        stand['aufrufe'] = stand['aufrufe'] + 1
        if len(frei) >= stand['beste']:      # nicht besser als die beste Lösung: abschneiden
            return
        if i == n:                           # alle Pakete verteilt, und weniger Wagen als bisher
            stand['beste'] = len(frei)
            return
        for w in range(len(frei)):           # auf einen vorhandenen Wagen
            if frei[w] >= pakete[i]:
                frei[w] = frei[w] - pakete[i]
                probiere(i + 1)
                frei[w] = frei[w] + pakete[i]
        frei.append(K - pakete[i])           # oder auf einen neuen Wagen
        probiere(i + 1)
        frei.pop()

    probiere(0)
    return stand['beste'], stand['aufrufe']

print(exakt(pakete, K))
print(exakt([4, 4, 4, 4], 8))
```

Das Optimum für die Pakete von oben ist **3 Wagen**, gefunden mit 30 Aufrufen. Damit ist First Fit optimal und First Fit Decreasing (4 Wagen) einen Wagen zu viel.
Bei `[4, 4, 4, 4]` und Kapazität 8 passen je zwei Pakete auf einen Wagen: 2 Wagen.

</details>

#### Frage 3.2

Für 8, 10, 12, 14 und 16 zufällige Pakete (Gewichte von 3 bis 12 kg, Kapazität 20) braucht `exakt` etwa 49, 308, 1 987, 16 732 und 39 359 Aufrufe.
1. Wie wächst die Zahl der Aufrufe etwa, wenn man zwei Pakete mehr hat?
2. Warum ist `exakt` für 60 Pakete **keine** Lösung, und wozu ist es trotzdem gut?

✏️ *Deine Antwort (Doppelklick zum Bearbeiten):*

<details>
<summary>Lösung anzeigen (erst selbst überlegen!)</summary>

1. Bei zwei Paketen mehr wächst sie ungefähr um den **Faktor 6 bis 8** (49, 308, 1 987, 16 732), pro Paket also etwa um den Faktor 2,5 bis 3. Das Wachstum ist **exponentiell**,
   die Laufzeit also etwa O(c^n) mit c größer als 2. Dass es bei 16 Paketen nur „39 359“ sind, liegt am Abschneiden, es hängt vom Fall ab.
2. Bei 60 Paketen sind es unvorstellbar viele Möglichkeiten, auch mit Abschneiden. Aber für **kleine** Fälle (bis etwa 14 Pakete) liefert `exakt` das **echte Optimum** als Vergleich:
   Man weiß so, wie weit eine Heuristik von der **besten möglichen** Lösung entfernt ist. Die Heuristik wird dann auf **großen** Fällen eingesetzt, und man vertraut dem, was man an den kleinen gesehen hat.

</details>

## 4. Untere Schranken: wie gut kann es höchstens gehen?

Jede Lösung, die der Schiedsrichter durchlässt, ist eine **obere Schranke** für das Optimum: Man schafft es mit so vielen Wagen, vielleicht aber mit weniger.
Eine **untere Schranke** ist eine Zahl, die man mit einer **Begründung** nicht unterschreiten kann: **Weniger Wagen gibt es nicht.** Zwei Schranken fürs Behälterproblem:

1. **Gewicht:** Alle Pakete zusammen wiegen `S` kg, ein Wagen trägt höchstens `K`. Mit `w` Wagen kann man höchstens `w · K` kg tragen. Also braucht man mindestens `S / K` Wagen,
   **aufgerundet** (Wagen kann man nicht teilen). Aufrunden geht ohne Bibliothek mit `-(-S // K)`: Die ganzzahlige Division `//` rundet **ab**, mit zwei Minuszeichen wird daraus „aufrunden“.
2. **Große Pakete:** Zwei Pakete, die **jeweils schwerer als `K/2`** sind, wiegen zusammen mehr als `K` und passen **nie** auf denselben Wagen. Jedes braucht einen **eigenen** Wagen.
   Also braucht man mindestens so viele Wagen, wie es solche Pakete gibt. In Python: `2 * p > K`, das vermeidet Brüche.

Beide gelten **für jede** Eingabe, und die bessere ist das **Maximum** aus beiden.

#### Aufgabe 4.1

Schreibe `schranke_gewicht`, `schranke_gross` und `schranke` (das Maximum der beiden). Probiere sie auch an `[11, 11, 11, 1, 1, 1]` und an acht Paketen zu je 5 kg, jeweils mit `K = 20`, aus.
Diese beiden zeigen, **warum** man beide Schranken braucht.

**Erwartete Ausgabe:**
```
3 0 3
3
2
```

In [ ]:
K = 20                                       # Kapazität eines Wagens in kg
pakete = [7, 9, 10, 7, 4, 7, 8, 3, 4]        # Gewichte der Pakete in kg

def schranke_gewicht(pakete, K):
    return 0                                 # dein Code: Gesamtgewicht durch K, aufgerundet


def schranke_gross(pakete, K):
    return 0                                 # dein Code: Pakete zählen, die schwerer als K / 2 sind


def schranke(pakete, K):
    return 0                                 # dein Code: das Maximum



print(schranke_gewicht(pakete, K), schranke_gross(pakete, K), schranke(pakete, K))
print(schranke([11, 11, 11, 1, 1, 1], 20))
print(schranke([5, 5, 5, 5, 5, 5, 5, 5], 20))

<details>
<summary>Tipp anzeigen</summary>

`-(-sum(pakete) // K)` rundet auf. In `schranke_gross` zählst du mit einer Schleife die Pakete mit `2 * p > K`. `max(a, b)` gibt die größere Zahl.

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
K = 20                                       # Kapazität eines Wagens in kg
pakete = [7, 9, 10, 7, 4, 7, 8, 3, 4]        # Gewichte der Pakete in kg

def schranke_gewicht(pakete, K):
    return -(-sum(pakete) // K)              # aufgerundet: S / K


def schranke_gross(pakete, K):
    anzahl = 0
    for p in pakete:
        if 2 * p > K:                        # schwerer als K / 2
            anzahl = anzahl + 1
    return anzahl


def schranke(pakete, K):
    return max(schranke_gewicht(pakete, K), schranke_gross(pakete, K))

print(schranke_gewicht(pakete, K), schranke_gross(pakete, K), schranke(pakete, K))
print(schranke([11, 11, 11, 1, 1, 1], 20))
print(schranke([5, 5, 5, 5, 5, 5, 5, 5], 20))
```

Für unsere Pakete: 59 kg bei Kapazität 20 ergibt aufgerundet 3. Kein Paket ist schwerer als 10 kg (das schwerste wiegt genau 10 kg), `schranke_gross` ist also 0 und die Schranke 3 stammt aus dem Gewicht.
Bei `[11, 11, 11, 1, 1, 1]` wiegen die Pakete 36 kg, das Gewicht liefert nur 2. Aber drei Pakete sind schwerer als 10 kg, also braucht man **mindestens 3 Wagen**.
Bei acht Paketen zu je 5 kg sind es 40 kg, also mindestens 2 Wagen. Hier liefert die große-Pakete-Schranke 0, das Gewicht trägt.

</details>

**Wann ist die Schranke das Optimum?** Gibt es eine **erlaubte** Lösung (der Schiedsrichter sagt „ok“) mit genau so vielen Wagen wie die untere Schranke, dann gibt es **keine bessere**,
und die Lösung ist **bewiesen optimal**. Bei unseren Paketen sind das die 3 Wagen von First Fit. Bei First Fit Decreasing mit 4 Wagen bleibt dagegen ein Abstand von einem Wagen.
Man weiß dann: Es gibt höchstens einen Wagen zu viel, aber man weiß **nicht**, ob die Schranke erreichbar ist.

#### Frage 4.2

Ein Schüler schreibt in `schranke_gross` statt `2 * p > K` die Bedingung `2 * p >= K` („mindestens halb so schwer wie K“). Gib ein **Beispiel**, bei dem seine „Schranke“ falsch (zu hoch) ist,
und erkläre, warum.

✏️ *Deine Antwort (Doppelklick zum Bearbeiten):*

<details>
<summary>Lösung anzeigen (erst selbst überlegen!)</summary>

`pakete = [10, 10, 10, 10]`, `K = 20`: Alle vier Pakete wiegen genau `K/2`, mit `>=` zählt er 4 und behauptet, man brauche mindestens 4 Wagen. Zwei Pakete mit **genau** 10 kg wiegen zusammen genau 20 kg
und passen auf **einen** Wagen. 2 Wagen reichen. Eine „untere Schranke“, die über dem Optimum liegt, ist **keine Schranke**, sie ist **falsch** und verdirbt jede Aussage über die Güte.
Bei Schranken immer den **Grenzfall** prüfen: Was passiert bei Gleichheit?

</details>

**Die Schranke ist nicht immer erreichbar.** Nimm Pakete `[6, 7, 4, 4, 4, 4]` und `K = 10`. Gewicht: 29 kg, also mindestens 3 Wagen. Große Pakete: höchstens 2 (6 und 7). Die Schranke ist 3.
Aber das Paket mit 7 kg hat nur 3 kg Platz neben sich, es bleibt allein. Das Paket mit 6 kg nimmt nur **ein** 4-kg-Paket mit, die anderen drei 4-kg-Pakete brauchen zwei weitere Wagen. Das sind 4 Wagen:

In [ ]:
K = 20                                       # Kapazität eines Wagens in kg
pakete = [7, 9, 10, 7, 4, 7, 8, 3, 4]        # Gewichte der Pakete in kg

def exakt(pakete, K):
    """Kleinste mögliche Zahl der Wagen und Zahl der Aufrufe."""
    n = len(pakete)
    frei = []                                # frei[w]: noch freie Kapazität von Wagen w
    stand = {'beste': n + 1, 'aufrufe': 0}   # ein Dictionary, damit die innere Funktion die Werte ändern kann

    def probiere(i):
        stand['aufrufe'] = stand['aufrufe'] + 1
        if len(frei) >= stand['beste']:      # nicht besser als die beste Lösung: abschneiden
            return
        if i == n:                           # alle Pakete verteilt, und weniger Wagen als bisher
            stand['beste'] = len(frei)
            return
        for w in range(len(frei)):           # auf einen vorhandenen Wagen
            if frei[w] >= pakete[i]:
                frei[w] = frei[w] - pakete[i]
                probiere(i + 1)
                frei[w] = frei[w] + pakete[i]
        frei.append(K - pakete[i])           # oder auf einen neuen Wagen
        probiere(i + 1)
        frei.pop()

    probiere(0)
    return stand['beste'], stand['aufrufe']

def schranke_gewicht(pakete, K):
    return -(-sum(pakete) // K)              # aufgerundet: S / K


def schranke_gross(pakete, K):
    anzahl = 0
    for p in pakete:
        if 2 * p > K:                        # schwerer als K / 2
            anzahl = anzahl + 1
    return anzahl


def schranke(pakete, K):
    return max(schranke_gewicht(pakete, K), schranke_gross(pakete, K))

luecke = [6, 7, 4, 4, 4, 4]
print('Schranke', schranke(luecke, 10), ' Optimum', exakt(luecke, 10)[0])

Schranke 3  Optimum 4


Hier liegt die beste Lösung **über** der Schranke. Es gibt Fälle, in denen man die Schranke gar nicht erreichen kann, und dann ist ein Abstand zur Schranke **nicht** dasselbe wie ein Abstand zum Optimum.
Genau darauf kommt es in der Dokumentation an (Abschnitt 5).

## 5. Die Güte messen: eine Tabelle über viele Zufallsfälle

Ein einzelnes Beispiel sagt wenig. Wir erzeugen **viele** Fälle mit festem Startwert (`random.Random(startwert)` wie in [Notebook 15](https://ktheu.github.io/bwinfN/notebooks/index.html?path=learn_bwinf1/15_ereignisse_zufall.ipynb) von learn_bwinf1,
dann sind alle Läufe gleich), lösen jeden mit jedem Verfahren und messen den **Abstand zur Schranke in Prozent**:

> **Güte** = (Wagen − Schranke) / Schranke · 100 %

Eine Güte von 0 % heißt: Das Verfahren hat die Schranke erreicht, die Lösung ist **bewiesen optimal**.

#### Aufgabe 5.1

Schreibe `zufallsfall(startwert, n, lo, hi)`: `n` ganze Zufallszahlen von `lo` bis `hi` (mit `randint`) als Liste. Der Zufallsgenerator entsteht **in** der Funktion
mit `random.Random(startwert)`, damit derselbe Startwert **immer** denselben Fall liefert.

**Erwartete Ausgabe:**
```
[18, 46, 58, 14, 26, 17, 41, 58, 38, 40]
True
[13, 15, 15, 33, 20, 57, 52, 29, 26, 48]
```

In [ ]:
import random


def zufallsfall(startwert, n, lo, hi):
    # dein Code
    return []


print(zufallsfall(1, 10, 10, 60))
print(zufallsfall(1, 10, 10, 60) == zufallsfall(1, 10, 10, 60))
print(zufallsfall(2, 10, 10, 60))

<details>
<summary>Tipp anzeigen</summary>

`zufall = random.Random(startwert)`, dann in einer Schleife `pakete.append(zufall.randint(lo, hi))`.

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
import random


def zufallsfall(startwert, n, lo, hi):
    """n Pakete mit Gewichten von lo bis hi, bei jedem Startwert dieselben."""
    zufall = random.Random(startwert)
    pakete = []
    for i in range(n):
        pakete.append(zufall.randint(lo, hi))
    return pakete

print(zufallsfall(1, 10, 10, 60))
print(zufallsfall(1, 10, 10, 60) == zufallsfall(1, 10, 10, 60))
print(zufallsfall(2, 10, 10, 60))
```

Gleicher Startwert, gleicher Fall (die zweite Zeile ist `True`), anderer Startwert, anderer Fall. So kann jeder deine Tabelle nachrechnen.

</details>

#### Aufgabe 5.2

Schreibe `guete_tabelle(verfahren_liste, n, anzahl)`. `verfahren_liste` ist eine Liste von Paaren `(name, funktion)`. Für jedes Verfahren löst sie die Fälle mit den Startwerten 1 bis `anzahl`
(`n` Pakete, Gewichte 10 bis 60, Kapazität 100) und gibt eine Zeile aus mit: wie oft die Schranke erreicht wurde, dem **mittleren** und dem **größten** Abstand zur Schranke in Prozent.
Benutze `bewerte` (mit Schiedsrichter!) und `schranke`. Die Ausgabe-Zeile ist schon fertig.

**Erwartete Ausgabe:**
```
20 Pakete, Gewichte 10 bis 60, K = 100
First Fit              Schranke erreicht: 108 von 200, im Mittel  6.8 % darüber, höchstens 28.6 %
First Fit Decreasing   Schranke erreicht: 164 von 200, im Mittel  2.5 % darüber, höchstens 16.7 %
40 Pakete, Gewichte 10 bis 60, K = 100
First Fit              Schranke erreicht:  39 von 200, im Mittel  6.4 % darüber, höchstens 15.4 %
First Fit Decreasing   Schranke erreicht: 144 von 200, im Mittel  2.0 % darüber, höchstens  8.3 %
80 Pakete, Gewichte 10 bis 60, K = 100
First Fit              Schranke erreicht:   1 von 200, im Mittel  5.5 % darüber, höchstens 10.7 %
First Fit Decreasing   Schranke erreicht: 112 von 200, im Mittel  1.5 % darüber, höchstens  3.8 %
```

In [ ]:
K = 20                                       # Kapazität eines Wagens in kg
pakete = [7, 9, 10, 7, 4, 7, 8, 3, 4]        # Gewichte der Pakete in kg

def pruefe(pakete, K, zuordnung):
    """Gibt 'ok' zurück oder beschreibt den ersten Verstoß."""
    if len(zuordnung) != len(pakete):
        return f'{len(zuordnung)} Einträge für {len(pakete)} Pakete'
    last = {}                                # last[w]: Gewicht auf Wagen w
    for i in range(len(pakete)):
        w = zuordnung[i]
        if w < 0:
            return f'Paket {i} hat die ungültige Wagennummer {w}'
        last[w] = last.get(w, 0) + pakete[i]
    for w in sorted(last):
        if last[w] > K:
            return f'Wagen {w} ist überladen: {last[w]} kg bei höchstens {K} kg'
    return 'ok'


def wagenzahl(zuordnung):
    return len(set(zuordnung))

def first_fit(pakete, K, reihenfolge):
    """reihenfolge: Liste der Paketnummern in der Reihenfolge, in der sie verteilt werden."""
    frei = []                                # frei[w]: noch freie Kapazität von Wagen w
    zuordnung = [0] * len(pakete)
    for i in reihenfolge:
        gefunden = -1
        for w in range(len(frei)):
            if frei[w] >= pakete[i]:
                gefunden = w
                break                        # der erste passende Wagen
        if gefunden == -1:                   # keiner passt: neuer Wagen
            frei.append(K)
            gefunden = len(frei) - 1
        frei[gefunden] = frei[gefunden] - pakete[i]
        zuordnung[i] = gefunden
    return zuordnung

def ff_gegeben(pakete, K):
    return first_fit(pakete, K, list(range(len(pakete))))


def ffd(pakete, K):
    # Paketnummern nach Gewicht absteigend: key=lambda wie in Notebook 03 von learn_bwinf1
    reihenfolge = sorted(range(len(pakete)), key=lambda i: -pakete[i])
    return first_fit(pakete, K, reihenfolge)

def bewerte(verfahren, pakete, K):
    """Führt verfahren(pakete, K) aus, lässt den Schiedsrichter prüfen und gibt die Zahl der Wagen zurück."""
    zuordnung = verfahren(pakete, K)
    urteil = pruefe(pakete, K, zuordnung)
    if urteil != 'ok':
        raise ValueError('Schiedsrichter: ' + urteil)
    return wagenzahl(zuordnung)

def schranke_gewicht(pakete, K):
    return -(-sum(pakete) // K)              # aufgerundet: S / K


def schranke_gross(pakete, K):
    anzahl = 0
    for p in pakete:
        if 2 * p > K:                        # schwerer als K / 2
            anzahl = anzahl + 1
    return anzahl


def schranke(pakete, K):
    return max(schranke_gewicht(pakete, K), schranke_gross(pakete, K))

import random


def zufallsfall(startwert, n, lo, hi):
    """n Pakete mit Gewichten von lo bis hi, bei jedem Startwert dieselben."""
    zufall = random.Random(startwert)
    pakete = []
    for i in range(n):
        pakete.append(zufall.randint(lo, hi))
    return pakete

def guete_tabelle(verfahren_liste, n, anzahl):
    for name, verfahren in verfahren_liste:
        gleich = 0
        summe = 0
        groesster = 0
        for startwert in range(1, anzahl + 1):
            pakete = zufallsfall(startwert, n, 10, 60)
            # dein Code: wagen, s und prozent berechnen; gleich, summe und groesster nachführen
            pass
        print(f'{name:22} Schranke erreicht: {gleich:3} von {anzahl}, im Mittel {summe / anzahl:4.1f} % darüber, höchstens {groesster:4.1f} %')


VERFAHREN = [('First Fit', ff_gegeben), ('First Fit Decreasing', ffd)]
for n in [20, 40, 80]:
    print(n, 'Pakete, Gewichte 10 bis 60, K = 100')
    guete_tabelle(VERFAHREN, n, 200)

<details>
<summary>Tipp anzeigen</summary>

`wagen = bewerte(verfahren, pakete, 100)`, `s = schranke(pakete, 100)`, `prozent = 100 * (wagen - s) / s`. Ist `wagen == s`, zähle `gleich` hoch.
`summe` sammelt die Prozentwerte (für den Mittelwert), `groesster = max(groesster, prozent)`.

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
K = 20                                       # Kapazität eines Wagens in kg
pakete = [7, 9, 10, 7, 4, 7, 8, 3, 4]        # Gewichte der Pakete in kg

def pruefe(pakete, K, zuordnung):
    """Gibt 'ok' zurück oder beschreibt den ersten Verstoß."""
    if len(zuordnung) != len(pakete):
        return f'{len(zuordnung)} Einträge für {len(pakete)} Pakete'
    last = {}                                # last[w]: Gewicht auf Wagen w
    for i in range(len(pakete)):
        w = zuordnung[i]
        if w < 0:
            return f'Paket {i} hat die ungültige Wagennummer {w}'
        last[w] = last.get(w, 0) + pakete[i]
    for w in sorted(last):
        if last[w] > K:
            return f'Wagen {w} ist überladen: {last[w]} kg bei höchstens {K} kg'
    return 'ok'


def wagenzahl(zuordnung):
    return len(set(zuordnung))

def first_fit(pakete, K, reihenfolge):
    """reihenfolge: Liste der Paketnummern in der Reihenfolge, in der sie verteilt werden."""
    frei = []                                # frei[w]: noch freie Kapazität von Wagen w
    zuordnung = [0] * len(pakete)
    for i in reihenfolge:
        gefunden = -1
        for w in range(len(frei)):
            if frei[w] >= pakete[i]:
                gefunden = w
                break                        # der erste passende Wagen
        if gefunden == -1:                   # keiner passt: neuer Wagen
            frei.append(K)
            gefunden = len(frei) - 1
        frei[gefunden] = frei[gefunden] - pakete[i]
        zuordnung[i] = gefunden
    return zuordnung

def ff_gegeben(pakete, K):
    return first_fit(pakete, K, list(range(len(pakete))))


def ffd(pakete, K):
    # Paketnummern nach Gewicht absteigend: key=lambda wie in Notebook 03 von learn_bwinf1
    reihenfolge = sorted(range(len(pakete)), key=lambda i: -pakete[i])
    return first_fit(pakete, K, reihenfolge)

def bewerte(verfahren, pakete, K):
    """Führt verfahren(pakete, K) aus, lässt den Schiedsrichter prüfen und gibt die Zahl der Wagen zurück."""
    zuordnung = verfahren(pakete, K)
    urteil = pruefe(pakete, K, zuordnung)
    if urteil != 'ok':
        raise ValueError('Schiedsrichter: ' + urteil)
    return wagenzahl(zuordnung)

def schranke_gewicht(pakete, K):
    return -(-sum(pakete) // K)              # aufgerundet: S / K


def schranke_gross(pakete, K):
    anzahl = 0
    for p in pakete:
        if 2 * p > K:                        # schwerer als K / 2
            anzahl = anzahl + 1
    return anzahl


def schranke(pakete, K):
    return max(schranke_gewicht(pakete, K), schranke_gross(pakete, K))

import random


def zufallsfall(startwert, n, lo, hi):
    """n Pakete mit Gewichten von lo bis hi, bei jedem Startwert dieselben."""
    zufall = random.Random(startwert)
    pakete = []
    for i in range(n):
        pakete.append(zufall.randint(lo, hi))
    return pakete

def guete_tabelle(verfahren_liste, n, anzahl):
    """Je Verfahren: wie oft die Schranke erreicht wurde, mittlerer und größter Abstand in Prozent."""
    for name, verfahren in verfahren_liste:
        gleich = 0
        summe = 0
        groesster = 0
        for startwert in range(1, anzahl + 1):
            pakete = zufallsfall(startwert, n, 10, 60)
            wagen = bewerte(verfahren, pakete, 100)
            s = schranke(pakete, 100)
            prozent = 100 * (wagen - s) / s
            if wagen == s:
                gleich = gleich + 1
            summe = summe + prozent
            groesster = max(groesster, prozent)
        print(f'{name:22} Schranke erreicht: {gleich:3} von {anzahl}, im Mittel {summe / anzahl:4.1f} % darüber, höchstens {groesster:4.1f} %')

VERFAHREN = [('First Fit', ff_gegeben), ('First Fit Decreasing', ffd)]
for n in [20, 40, 80]:
    print(n, 'Pakete, Gewichte 10 bis 60, K = 100')
    guete_tabelle(VERFAHREN, n, 200)
```

Beide Verfahren sehen in der Tabelle sehr verschieden aus, und der Unterschied wächst mit der Zahl der Pakete.

</details>

So liest du die Tabelle:

- Bei **40 Paketen** erreicht First Fit die Schranke in nur **39 von 200** Fällen, im Mittel liegt es 6,4 % darüber, im schlechtesten Fall 15,4 %.
  First Fit Decreasing erreicht sie in **144 von 200** Fällen, im Mittel 2,0 % darüber, im schlechtesten Fall 8,3 %.
- Bei **80 Paketen** sind es 1 (First Fit) und 112 (First Fit Decreasing) von 200. Mehr Pakete heißt: viele kleine Lücken, die man mit **sortierten** Paketen besser füllt.
- Ein Wagen mehr bei 20 Wagen sind schon 5 %. Darum sind die Prozentwerte bei **kleinen** Wagenzahlen sehr grob (ein Wagen mehr sind dort schnell 20 bis 30 %).

Was die „Schranke erreicht“-Spalte sagt: In diesen Fällen ist die Lösung **bewiesen optimal**. In den übrigen Fällen wissen wir nur: Die Lösung liegt höchstens so viele Prozent über dem Optimum,
wie die Tabelle für den Abstand zur Schranke angibt. Ob sie wirklich besser sein könnte, bleibt offen (Abschnitt 4: Die Schranke ist nicht immer erreichbar).

**Wie gut ist die Schranke selbst?** Bei **kleinen** Fällen kennen wir mit `exakt` das echte Optimum und können alles vergleichen: die Schranke, First Fit und First Fit Decreasing (100 Fälle mit je 10 Paketen, Gewichte 3 bis 12, `K = 20`):

In [ ]:
anzahl_faelle = 100
schranke_ist_optimum = 0
ff_ist_optimum = 0
ffd_ist_optimum = 0
for startwert in range(1, anzahl_faelle + 1):
    pakete_klein = zufallsfall(startwert, 10, 3, 12)
    optimum = exakt(pakete_klein, 20)[0]
    if schranke(pakete_klein, 20) == optimum:
        schranke_ist_optimum = schranke_ist_optimum + 1
    if bewerte(ff_gegeben, pakete_klein, 20) == optimum:
        ff_ist_optimum = ff_ist_optimum + 1
    if bewerte(ffd, pakete_klein, 20) == optimum:
        ffd_ist_optimum = ffd_ist_optimum + 1
print('von', anzahl_faelle, 'kleinen Fällen (10 Pakete, K = 20):')
print('Schranke ist das Optimum:', schranke_ist_optimum)
print('First Fit ist optimal:', ff_ist_optimum)
print('First Fit Decreasing ist optimal:', ffd_ist_optimum)

von 100 kleinen Fällen (10 Pakete, K = 20):
Schranke ist das Optimum: 94
First Fit ist optimal: 79
First Fit Decreasing ist optimal: 94


In **94 von 100** Fällen ist die Schranke genau das Optimum, die Schranke ist hier also **meist ehrlich**. First Fit ist in 79 und First Fit Decreasing in 94 Fällen optimal.
Bei kleinen Fällen kann man das Optimum berechnen, bei großen nicht mehr. Dort bleibt die **Schranke** als Maßstab.

#### Frage 5.3

Die Tabelle zeigt für 40 Pakete: First Fit Decreasing liegt im schlechtesten Fall 8,3 % über der Schranke. Was darfst du in deiner Dokumentation schreiben, und was **nicht**?
Wähle aus:

1. „Das Verfahren liefert immer höchstens 8,3 % mehr Wagen als nötig.“
2. „In 200 Zufallsfällen mit 40 Paketen lag das Verfahren höchstens 8,3 % über der unteren Schranke, also höchstens 8,3 % über dem Optimum.“
3. „Das Verfahren liegt im Mittel 3,3 % neben dem Optimum.“

✏️ *Deine Antwort (Doppelklick zum Bearbeiten):*

<details>
<summary>Lösung anzeigen (erst selbst überlegen!)</summary>

Nur **2** ist richtig. Das Optimum ist **mindestens** so groß wie die Schranke, also ist der Abstand zum Optimum **höchstens** so groß wie der zur Schranke. „Höchstens x % über dem Optimum“ ist daher ehrlich,
auch wenn man das Optimum nicht kennt. Wichtig ist, **was gemessen wurde** (200 Zufallsfälle, Gewichte 10 bis 60, `K = 100`, Startwerte 1 bis 200).

**1** ist keine Messung, sondern eine **Garantie** für alle Eingaben. Dafür bräuchte man einen Beweis, 200 Fälle reichen nicht. Auch wenn sie nach 200 Fällen plausibel klingt, ist sie nicht gedeckt.
**3** verwechselt den Abstand zur **Schranke** (den wir kennen) mit dem zum **Optimum** (das wir nicht kennen). Der echte Abstand zum Optimum ist **höchstens** so groß, nicht genau so groß.

</details>

## Wettbewerbs-Ecke 1: Drohnen werfen Pakete

*Angelehnt an Aufgabe 2 „Mißglückte Drohnenlieferung“ der 2. Runde des 34. Bundeswettbewerbs Informatik (2015/16).*

In einer Siedlung stehen `n × n` Häuser im Quadrat. Auf jedem Haus liegt **ein** Paket, aber das falsche: Auf dem Haus in Zeile `i` und Spalte `j` liegt das Paket für das Haus `(a, b)`.
Ein **Plan** besteht aus **Schritten**. In einem Schritt nimmt jede Person ihr Paket und wirft es **gleichzeitig** nach Norden (`N`), Süden (`S`), Osten (`O`) oder Westen (`W`) zum Nachbarhaus oder behält es.
**Nach jedem Schritt muss auf jedem Haus genau ein Paket liegen.** Am Ende muss jedes Paket auf seinem Haus liegen. Gesucht ist ein Plan mit **möglichst wenigen Schritten**.

In `daten/` liegen drei Siedlungen (Dateien `drohnen_eingabe_*.txt`: erste Zeile `n`, dann je Haus eine Zeile `i j a b`):

- `beispiel`: 3 × 3 Häuser mit dem Plan der Aufgabe (`drohnen_plan_beispiel.txt`, je Haus eine Zeile `i j` und ein Buchstabe pro Schritt, `_` heißt: nicht werfen),
- `stadt5`: 5 × 5 Häuser, eine Zufallsverteilung,
- `amacity`: 10 × 10 Häuser, die Verteilung der echten Aufgabe (nur für Profis).

Die Kernbotschaft dieser Ecke: **Der Schiedsrichter prüft jeden Plan, und die Manhattan-Schranke sagt, wie gut ein Plan höchstens sein kann.**

#### Aufgabe E1.1

Der Schiedsrichter für Pläne besteht aus zwei Teilen. `pruefe_plan` und `anzahl_wuerfe` sind schon fertig. Du ergänzt in `fuehre_schritt_aus` die **beiden Regeln**, die in einem Schritt verletzt sein können:
Ein Paket fliegt über den **Rand** der Siedlung hinaus, und **zwei Pakete** landen auf demselben Haus. Bei einem Verstoß gibt die Funktion einen **Text** zurück (der Fehler wird in `pruefe_plan` mit
der Nummer des Schrittes weitergegeben), sonst die neue Lage.
Die Aufgabe liefert noch kein Verfahren, sondern **prüft vorhandene Pläne**: den Plan der Aufgabe und drei absichtlich falsche.

**Erwartete Ausgabe:**
```
4 Schritte, 20 Würfe: ok
Fehler in Schritt 1: auf Haus (1, 0) landen zwei Pakete
Fehler in Schritt 1: Haus (0, 0) wirft aus der Siedlung hinaus
am Ende liegen 5 Pakete falsch
```

In [ ]:
RICHTUNG = {'N': (-1, 0), 'S': (1, 0), 'O': (0, 1), 'W': (0, -1)}


def lies_stadt(dateiname):
    """Gibt n und das Dictionary lage zurück: lage[(i, j)] = (a, b) heißt: auf Haus (i, j) liegt das Paket für Haus (a, b)."""
    with open('daten/' + dateiname, encoding='utf-8') as datei:
        zeilen = datei.read().split('\n')
    n = int(zeilen[0])
    lage = {}
    for zeile in zeilen[1:]:
        teile = zeile.split()
        if len(teile) == 4:
            i, j, a, b = [int(x) for x in teile]
            lage[(i, j)] = (a, b)
    return n, lage


def lies_plan(dateiname):
    """Ein Plan ist eine Liste von Schritten, ein Schritt ein Dictionary {Haus: Richtung} der werfenden Häuser."""
    zeilen = {}
    with open('daten/' + dateiname, encoding='utf-8') as datei:
        for zeile in datei.read().split('\n'):
            teile = zeile.split()
            if len(teile) == 3:
                zeilen[(int(teile[0]), int(teile[1]))] = teile[2]
    plan = []
    for k in range(len(zeilen[(0, 0)])):
        schritt = {}
        for haus in zeilen:
            if zeilen[haus][k] != '_':
                schritt[haus] = zeilen[haus][k]
        plan.append(schritt)
    return plan

def fuehre_schritt_aus(n, lage, schritt):
    neu = {}
    for haus in lage:
        if haus in schritt:
            dz, ds = RICHTUNG[schritt[haus]]
            wohin = (haus[0] + dz, haus[1] + ds)
            # dein Code 1: liegt wohin außerhalb von 0 bis n - 1? Dann einen Text zurückgeben
        else:
            wohin = haus                         # wer nicht wirft, behält sein Paket
        # dein Code 2: liegt auf wohin schon ein Paket (wohin in neu)? Dann einen Text zurückgeben
        neu[wohin] = lage[haus]
    return neu

def pruefe_plan(n, lage, plan):
    for k in range(len(plan)):
        lage = fuehre_schritt_aus(n, lage, plan[k])
        if isinstance(lage, str):                # ein Text: der Schritt war nicht erlaubt
            return f'Fehler in Schritt {k + 1}: {lage}'
    falsch = 0
    for haus in lage:
        if lage[haus] != haus:
            falsch = falsch + 1
    if falsch > 0:
        return f'am Ende liegen {falsch} Pakete falsch'
    return 'ok'


def anzahl_wuerfe(plan):
    summe = 0
    for schritt in plan:
        summe = summe + len(schritt)
    return summe

n, lage = lies_stadt('drohnen_eingabe_beispiel.txt')
plan = lies_plan('drohnen_plan_beispiel.txt')
print(len(plan), 'Schritte,', anzahl_wuerfe(plan), 'Würfe:', pruefe_plan(n, lage, plan))
print(pruefe_plan(n, lage, [{(0, 0): 'S', (2, 0): 'N'}]))
print(pruefe_plan(n, lage, [{(0, 0): 'N'}]))
print(pruefe_plan(n, lage, [{(0, 0): 'O', (0, 1): 'W'}]))

<details>
<summary>Tipp anzeigen</summary>

1. `if not (0 <= wohin[0] < n and 0 <= wohin[1] < n): return f'Haus {haus} wirft aus der Siedlung hinaus'`
2. `if wohin in neu: return f'auf Haus {wohin} landen zwei Pakete'`. Das Dictionary `neu` sammelt, welches Paket wohin kommt, ein zweiter Eintrag am selben Haus ist ein Verstoß.

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
RICHTUNG = {'N': (-1, 0), 'S': (1, 0), 'O': (0, 1), 'W': (0, -1)}


def lies_stadt(dateiname):
    """Gibt n und das Dictionary lage zurück: lage[(i, j)] = (a, b) heißt: auf Haus (i, j) liegt das Paket für Haus (a, b)."""
    with open('daten/' + dateiname, encoding='utf-8') as datei:
        zeilen = datei.read().split('\n')
    n = int(zeilen[0])
    lage = {}
    for zeile in zeilen[1:]:
        teile = zeile.split()
        if len(teile) == 4:
            i, j, a, b = [int(x) for x in teile]
            lage[(i, j)] = (a, b)
    return n, lage


def lies_plan(dateiname):
    """Ein Plan ist eine Liste von Schritten, ein Schritt ein Dictionary {Haus: Richtung} der werfenden Häuser."""
    zeilen = {}
    with open('daten/' + dateiname, encoding='utf-8') as datei:
        for zeile in datei.read().split('\n'):
            teile = zeile.split()
            if len(teile) == 3:
                zeilen[(int(teile[0]), int(teile[1]))] = teile[2]
    plan = []
    for k in range(len(zeilen[(0, 0)])):
        schritt = {}
        for haus in zeilen:
            if zeilen[haus][k] != '_':
                schritt[haus] = zeilen[haus][k]
        plan.append(schritt)
    return plan

def fuehre_schritt_aus(n, lage, schritt):
    """Gibt die neue Lage zurück oder einen Text, wenn der Schritt nicht erlaubt ist."""
    neu = {}
    for haus in lage:
        if haus in schritt:
            dz, ds = RICHTUNG[schritt[haus]]
            wohin = (haus[0] + dz, haus[1] + ds)
            if not (0 <= wohin[0] < n and 0 <= wohin[1] < n):
                return f'Haus {haus} wirft aus der Siedlung hinaus'
        else:
            wohin = haus                         # wer nicht wirft, behält sein Paket
        if wohin in neu:
            return f'auf Haus {wohin} landen zwei Pakete'
        neu[wohin] = lage[haus]
    return neu

def pruefe_plan(n, lage, plan):
    for k in range(len(plan)):
        lage = fuehre_schritt_aus(n, lage, plan[k])
        if isinstance(lage, str):                # ein Text: der Schritt war nicht erlaubt
            return f'Fehler in Schritt {k + 1}: {lage}'
    falsch = 0
    for haus in lage:
        if lage[haus] != haus:
            falsch = falsch + 1
    if falsch > 0:
        return f'am Ende liegen {falsch} Pakete falsch'
    return 'ok'


def anzahl_wuerfe(plan):
    summe = 0
    for schritt in plan:
        summe = summe + len(schritt)
    return summe

n, lage = lies_stadt('drohnen_eingabe_beispiel.txt')
plan = lies_plan('drohnen_plan_beispiel.txt')
print(len(plan), 'Schritte,', anzahl_wuerfe(plan), 'Würfe:', pruefe_plan(n, lage, plan))
print(pruefe_plan(n, lage, [{(0, 0): 'S', (2, 0): 'N'}]))
print(pruefe_plan(n, lage, [{(0, 0): 'N'}]))
print(pruefe_plan(n, lage, [{(0, 0): 'O', (0, 1): 'W'}]))
```

Der Plan der Aufgabe hat 4 Schritte mit 20 Würfen und besteht die Prüfung. Von den falschen Plänen wirft der erste von Haus (0, 0) nach Süden und von (2, 0) nach Norden: beide wollen auf Haus (1, 0),
das nicht wirft und sein Paket behält. Der zweite wirft über den Rand, der dritte tauscht nur zwei Pakete, danach liegen noch 5 falsch. Man sieht an der Meldung gleich, **wo** der Plan scheitert.

</details>

#### Aufgabe E1.2

Ein Paket kommt in jedem Schritt **höchstens ein Haus weit**. Der **Abstand** eines Pakets zu seinem Ziel ist `|Δzeile| + |Δspalte|` (die **Manhattan-Entfernung**, wie in einer Stadt mit rechtwinkligen Straßen).
Schreibe `abstand(haus, ziel)` und `untere_schranke(lage)`: Sie gibt die Zahl der Schritte zurück, die **jeder** Plan mindestens braucht.

**Erwartete Ausgabe:**
```
beispiel 4
stadt5 7
amacity 15
```

In [ ]:
RICHTUNG = {'N': (-1, 0), 'S': (1, 0), 'O': (0, 1), 'W': (0, -1)}


def lies_stadt(dateiname):
    """Gibt n und das Dictionary lage zurück: lage[(i, j)] = (a, b) heißt: auf Haus (i, j) liegt das Paket für Haus (a, b)."""
    with open('daten/' + dateiname, encoding='utf-8') as datei:
        zeilen = datei.read().split('\n')
    n = int(zeilen[0])
    lage = {}
    for zeile in zeilen[1:]:
        teile = zeile.split()
        if len(teile) == 4:
            i, j, a, b = [int(x) for x in teile]
            lage[(i, j)] = (a, b)
    return n, lage


def lies_plan(dateiname):
    """Ein Plan ist eine Liste von Schritten, ein Schritt ein Dictionary {Haus: Richtung} der werfenden Häuser."""
    zeilen = {}
    with open('daten/' + dateiname, encoding='utf-8') as datei:
        for zeile in datei.read().split('\n'):
            teile = zeile.split()
            if len(teile) == 3:
                zeilen[(int(teile[0]), int(teile[1]))] = teile[2]
    plan = []
    for k in range(len(zeilen[(0, 0)])):
        schritt = {}
        for haus in zeilen:
            if zeilen[haus][k] != '_':
                schritt[haus] = zeilen[haus][k]
        plan.append(schritt)
    return plan

def abstand(haus, ziel):
    return 0                                     # dein Code


def untere_schranke(lage):
    return 0                                     # dein Code


for name in ['beispiel', 'stadt5', 'amacity']:
    n, lage = lies_stadt('drohnen_eingabe_' + name + '.txt')
    print(name, untere_schranke(lage))

<details>
<summary>Tipp anzeigen</summary>

`abs(...)` ist der Betrag. Die Schranke ist der **größte** Abstand aller Pakete (`max`).

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
RICHTUNG = {'N': (-1, 0), 'S': (1, 0), 'O': (0, 1), 'W': (0, -1)}


def lies_stadt(dateiname):
    """Gibt n und das Dictionary lage zurück: lage[(i, j)] = (a, b) heißt: auf Haus (i, j) liegt das Paket für Haus (a, b)."""
    with open('daten/' + dateiname, encoding='utf-8') as datei:
        zeilen = datei.read().split('\n')
    n = int(zeilen[0])
    lage = {}
    for zeile in zeilen[1:]:
        teile = zeile.split()
        if len(teile) == 4:
            i, j, a, b = [int(x) for x in teile]
            lage[(i, j)] = (a, b)
    return n, lage


def lies_plan(dateiname):
    """Ein Plan ist eine Liste von Schritten, ein Schritt ein Dictionary {Haus: Richtung} der werfenden Häuser."""
    zeilen = {}
    with open('daten/' + dateiname, encoding='utf-8') as datei:
        for zeile in datei.read().split('\n'):
            teile = zeile.split()
            if len(teile) == 3:
                zeilen[(int(teile[0]), int(teile[1]))] = teile[2]
    plan = []
    for k in range(len(zeilen[(0, 0)])):
        schritt = {}
        for haus in zeilen:
            if zeilen[haus][k] != '_':
                schritt[haus] = zeilen[haus][k]
        plan.append(schritt)
    return plan

def abstand(haus, ziel):
    return abs(haus[0] - ziel[0]) + abs(haus[1] - ziel[1])


def untere_schranke(lage):
    """Mindestzahl der Schritte: der größte Abstand eines Pakets zu seinem Ziel."""
    groesster = 0
    for haus in lage:
        groesster = max(groesster, abstand(haus, lage[haus]))
    return groesster

for name in ['beispiel', 'stadt5', 'amacity']:
    n, lage = lies_stadt('drohnen_eingabe_' + name + '.txt')
    print(name, untere_schranke(lage))
```

Für `beispiel` ist die Schranke 4: Das Paket für Haus (0, 0) liegt auf Haus (2, 2), also 2 + 2 Häuser entfernt. Der Plan der Aufgabe hat 4 Schritte und ist damit **bewiesen optimal**.
Für `stadt5` sind es 7 Schritte, für `amacity` 15.

</details>

#### Frage E1.3

1. Warum ist der **größte** Abstand eine untere Schranke für die **Schritte**, die **Summe** der Abstände aber keine?
2. Wofür wäre die Summe der Abstände eine untere Schranke?
3. Reicht der größte Abstand, um zu **beweisen**, dass ein Plan optimal ist? Wann nicht?

✏️ *Deine Antwort (Doppelklick zum Bearbeiten):*

<details>
<summary>Lösung anzeigen (erst selbst überlegen!)</summary>

1. Alle Pakete werden **gleichzeitig** geworfen. Das am weitesten entfernte Paket kommt pro Schritt ein Haus näher, also braucht der Plan mindestens so viele Schritte, wie dieses Paket Abstand hat.
   Die Summe wäre viel größer, weil viele Pakete im selben Schritt vorankommen. Für das Beispiel: Summe 12, aber nur 4 Schritte nötig.
2. Für die **Zahl der Würfe** (jeder Wurf bringt ein Paket um ein Haus voran). Hat ein Plan genau so viele Würfe, wurde nie ein Paket vom Ziel weg geworfen.
3. Hat ein erlaubter Plan **genau** so viele Schritte wie die Schranke, ist er optimal. Hat er mehr, wissen wir nur: höchstens so viel zu viel. Es gibt Verteilungen, bei denen die Schranke nicht erreichbar ist
   (Beispiel: In einer Spalte mit drei Häusern müssen das obere und das untere Paket ihre Plätze tauschen. Die Schranke ist 2, nötig sind 3 Schritte, siehe „Engstelle“ im [Lösungsnotebook](https://ktheu.github.io/bwinfN/notebooks/index.html?path=bwinf2/34_MissglueckteDrohnenlieferung/drohnen_loesung.ipynb)).

</details>

#### Aufgabe E1.4

Jetzt ein einfaches **gieriges** Verfahren. Pro Schritt tauschen Nachbarhäuser ihre Pakete, wenn sich das lohnt. Die **Kosten** eines Pakets sind `d³` für den Abstand `d` zum Ziel (so zählen weit entfernte Pakete viel mehr).
Ein Tausch von `a` und `b` hat den **Gewinn** `vorher − nachher` (Kosten der beiden Pakete vor und nach dem Tausch). Nur Tausche mit Gewinn über 0 kommen in Frage. Aus allen wählt man **gierig** die mit dem größten
Gewinn, aber jedes Haus darf **nur in einem** Tausch vorkommen (sonst ist der Schritt nicht erlaubt). Schreibe `schritt_tauschen`, `finde_plan` und `teste` sind fertig.
`finde_plan` bekommt das Verfahren als **Parameter** `naechster_schritt` (eine Funktion, wie in Abschnitt 2).

Jeder Plan geht durch den Schiedsrichter. Die Ausgabe nennt auch die Schranke aus Aufgabe E1.2.

**Erwartete Ausgabe:**
```
beispiel: Schranke 4, fertig nach 4 Schritten mit 14 Würfen, Schiedsrichter: ok
stadt5: Schranke 7, steckt nach 10 Schritten fest, 22 Pakete liegen falsch, größter Abstand 3
amacity: Schranke 15, steckt nach 22 Schritten fest, 85 Pakete liegen falsch, größter Abstand 4
```

In [ ]:
RICHTUNG = {'N': (-1, 0), 'S': (1, 0), 'O': (0, 1), 'W': (0, -1)}


def lies_stadt(dateiname):
    """Gibt n und das Dictionary lage zurück: lage[(i, j)] = (a, b) heißt: auf Haus (i, j) liegt das Paket für Haus (a, b)."""
    with open('daten/' + dateiname, encoding='utf-8') as datei:
        zeilen = datei.read().split('\n')
    n = int(zeilen[0])
    lage = {}
    for zeile in zeilen[1:]:
        teile = zeile.split()
        if len(teile) == 4:
            i, j, a, b = [int(x) for x in teile]
            lage[(i, j)] = (a, b)
    return n, lage


def lies_plan(dateiname):
    """Ein Plan ist eine Liste von Schritten, ein Schritt ein Dictionary {Haus: Richtung} der werfenden Häuser."""
    zeilen = {}
    with open('daten/' + dateiname, encoding='utf-8') as datei:
        for zeile in datei.read().split('\n'):
            teile = zeile.split()
            if len(teile) == 3:
                zeilen[(int(teile[0]), int(teile[1]))] = teile[2]
    plan = []
    for k in range(len(zeilen[(0, 0)])):
        schritt = {}
        for haus in zeilen:
            if zeilen[haus][k] != '_':
                schritt[haus] = zeilen[haus][k]
        plan.append(schritt)
    return plan

def fuehre_schritt_aus(n, lage, schritt):
    """Gibt die neue Lage zurück oder einen Text, wenn der Schritt nicht erlaubt ist."""
    neu = {}
    for haus in lage:
        if haus in schritt:
            dz, ds = RICHTUNG[schritt[haus]]
            wohin = (haus[0] + dz, haus[1] + ds)
            if not (0 <= wohin[0] < n and 0 <= wohin[1] < n):
                return f'Haus {haus} wirft aus der Siedlung hinaus'
        else:
            wohin = haus                         # wer nicht wirft, behält sein Paket
        if wohin in neu:
            return f'auf Haus {wohin} landen zwei Pakete'
        neu[wohin] = lage[haus]
    return neu

def pruefe_plan(n, lage, plan):
    for k in range(len(plan)):
        lage = fuehre_schritt_aus(n, lage, plan[k])
        if isinstance(lage, str):                # ein Text: der Schritt war nicht erlaubt
            return f'Fehler in Schritt {k + 1}: {lage}'
    falsch = 0
    for haus in lage:
        if lage[haus] != haus:
            falsch = falsch + 1
    if falsch > 0:
        return f'am Ende liegen {falsch} Pakete falsch'
    return 'ok'


def anzahl_wuerfe(plan):
    summe = 0
    for schritt in plan:
        summe = summe + len(schritt)
    return summe

def abstand(haus, ziel):
    return abs(haus[0] - ziel[0]) + abs(haus[1] - ziel[1])


def untere_schranke(lage):
    """Mindestzahl der Schritte: der größte Abstand eines Pakets zu seinem Ziel."""
    groesster = 0
    for haus in lage:
        groesster = max(groesster, abstand(haus, lage[haus]))
    return groesster

def kosten(d):
    return d ** 3                                # weit entfernte Pakete zählen viel mehr (Vorrang)


def paketkosten(haus, ziel):
    return kosten(abstand(haus, ziel))


def gesamtkosten(lage):
    summe = 0
    for haus in lage:
        summe = summe + paketkosten(haus, lage[haus])
    return summe


def richtung(von, nach):
    for buchstabe in RICHTUNG:
        dz, ds = RICHTUNG[buchstabe]
        if (von[0] + dz, von[1] + ds) == nach:
            return buchstabe



def schritt_tauschen(n, lage):
    kandidaten = []
    for a in lage:
        for b in [(a[0], a[1] + 1), (a[0] + 1, a[1])]:       # rechter und unterer Nachbar: jedes Paar nur einmal
            if b in lage:
                pass                                         # dein Code: vorher, nachher, bei Gewinn merken (gewinn, a, b)
    schritt = {}
    # dein Code: größten Gewinn zuerst; nur nehmen, wenn a und b noch nicht in schritt vorkommen
    return schritt

def finde_plan(n, lage, naechster_schritt):
    """Wendet das übergebene Verfahren an, bis alles am Ziel ist oder die Kosten nicht mehr sinken."""
    plan = []
    while untere_schranke(lage) > 0 and len(plan) < 200:
        schritt = naechster_schritt(n, lage)
        neu = fuehre_schritt_aus(n, lage, schritt)
        if isinstance(neu, str) or gesamtkosten(neu) >= gesamtkosten(lage):
            break                                # kein erlaubter Schritt oder kein Fortschritt: steckt fest
        plan.append(schritt)
        lage = neu
    return plan, lage


def teste(namen, naechster_schritt):
    for name in namen:
        n, lage = lies_stadt('drohnen_eingabe_' + name + '.txt')
        plan, ende = finde_plan(n, lage, naechster_schritt)
        falsch = 0
        for haus in ende:
            if ende[haus] != haus:
                falsch = falsch + 1
        if falsch == 0:
            print(f'{name}: Schranke {untere_schranke(lage)}, fertig nach {len(plan)} Schritten mit {anzahl_wuerfe(plan)} Würfen, Schiedsrichter: {pruefe_plan(n, lage, plan)}')
        else:
            print(f'{name}: Schranke {untere_schranke(lage)}, steckt nach {len(plan)} Schritten fest, {falsch} Pakete liegen falsch, größter Abstand {untere_schranke(ende)}')

teste(['beispiel', 'stadt5', 'amacity'], schritt_tauschen)

<details>
<summary>Tipp anzeigen</summary>

`vorher = paketkosten(a, lage[a]) + paketkosten(b, lage[b])` und `nachher = paketkosten(b, lage[a]) + paketkosten(a, lage[b])` (das Paket von `a` liegt nach dem Tausch auf `b`). Ist `nachher < vorher`:
`kandidaten.append((vorher - nachher, a, b))`. Dann `for gewinn, a, b in sorted(kandidaten, reverse=True):` und `schritt[a] = richtung(a, b)`, `schritt[b] = richtung(b, a)`, wenn weder `a` noch `b` schon in `schritt` stehen.

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
RICHTUNG = {'N': (-1, 0), 'S': (1, 0), 'O': (0, 1), 'W': (0, -1)}


def lies_stadt(dateiname):
    """Gibt n und das Dictionary lage zurück: lage[(i, j)] = (a, b) heißt: auf Haus (i, j) liegt das Paket für Haus (a, b)."""
    with open('daten/' + dateiname, encoding='utf-8') as datei:
        zeilen = datei.read().split('\n')
    n = int(zeilen[0])
    lage = {}
    for zeile in zeilen[1:]:
        teile = zeile.split()
        if len(teile) == 4:
            i, j, a, b = [int(x) for x in teile]
            lage[(i, j)] = (a, b)
    return n, lage


def lies_plan(dateiname):
    """Ein Plan ist eine Liste von Schritten, ein Schritt ein Dictionary {Haus: Richtung} der werfenden Häuser."""
    zeilen = {}
    with open('daten/' + dateiname, encoding='utf-8') as datei:
        for zeile in datei.read().split('\n'):
            teile = zeile.split()
            if len(teile) == 3:
                zeilen[(int(teile[0]), int(teile[1]))] = teile[2]
    plan = []
    for k in range(len(zeilen[(0, 0)])):
        schritt = {}
        for haus in zeilen:
            if zeilen[haus][k] != '_':
                schritt[haus] = zeilen[haus][k]
        plan.append(schritt)
    return plan

def fuehre_schritt_aus(n, lage, schritt):
    """Gibt die neue Lage zurück oder einen Text, wenn der Schritt nicht erlaubt ist."""
    neu = {}
    for haus in lage:
        if haus in schritt:
            dz, ds = RICHTUNG[schritt[haus]]
            wohin = (haus[0] + dz, haus[1] + ds)
            if not (0 <= wohin[0] < n and 0 <= wohin[1] < n):
                return f'Haus {haus} wirft aus der Siedlung hinaus'
        else:
            wohin = haus                         # wer nicht wirft, behält sein Paket
        if wohin in neu:
            return f'auf Haus {wohin} landen zwei Pakete'
        neu[wohin] = lage[haus]
    return neu

def pruefe_plan(n, lage, plan):
    for k in range(len(plan)):
        lage = fuehre_schritt_aus(n, lage, plan[k])
        if isinstance(lage, str):                # ein Text: der Schritt war nicht erlaubt
            return f'Fehler in Schritt {k + 1}: {lage}'
    falsch = 0
    for haus in lage:
        if lage[haus] != haus:
            falsch = falsch + 1
    if falsch > 0:
        return f'am Ende liegen {falsch} Pakete falsch'
    return 'ok'


def anzahl_wuerfe(plan):
    summe = 0
    for schritt in plan:
        summe = summe + len(schritt)
    return summe

def abstand(haus, ziel):
    return abs(haus[0] - ziel[0]) + abs(haus[1] - ziel[1])


def untere_schranke(lage):
    """Mindestzahl der Schritte: der größte Abstand eines Pakets zu seinem Ziel."""
    groesster = 0
    for haus in lage:
        groesster = max(groesster, abstand(haus, lage[haus]))
    return groesster

def kosten(d):
    return d ** 3                                # weit entfernte Pakete zählen viel mehr (Vorrang)


def paketkosten(haus, ziel):
    return kosten(abstand(haus, ziel))


def gesamtkosten(lage):
    summe = 0
    for haus in lage:
        summe = summe + paketkosten(haus, lage[haus])
    return summe


def richtung(von, nach):
    for buchstabe in RICHTUNG:
        dz, ds = RICHTUNG[buchstabe]
        if (von[0] + dz, von[1] + ds) == nach:
            return buchstabe


def schritt_tauschen(n, lage):
    """Ein Schritt aus gleichzeitigen Tauschen: Nachbarn, die ihre Pakete tauschen, wenn die Kosten sinken."""
    kandidaten = []
    for a in lage:
        for b in [(a[0], a[1] + 1), (a[0] + 1, a[1])]:       # rechter und unterer Nachbar: jedes Paar nur einmal
            if b in lage:
                vorher = paketkosten(a, lage[a]) + paketkosten(b, lage[b])
                nachher = paketkosten(b, lage[a]) + paketkosten(a, lage[b])
                if nachher < vorher:
                    kandidaten.append((vorher - nachher, a, b))
    schritt = {}
    for gewinn, a, b in sorted(kandidaten, reverse=True):   # größter Gewinn zuerst
        if a not in schritt and b not in schritt:           # beide Häuser noch frei
            schritt[a] = richtung(a, b)
            schritt[b] = richtung(b, a)
    return schritt

def finde_plan(n, lage, naechster_schritt):
    """Wendet das übergebene Verfahren an, bis alles am Ziel ist oder die Kosten nicht mehr sinken."""
    plan = []
    while untere_schranke(lage) > 0 and len(plan) < 200:
        schritt = naechster_schritt(n, lage)
        neu = fuehre_schritt_aus(n, lage, schritt)
        if isinstance(neu, str) or gesamtkosten(neu) >= gesamtkosten(lage):
            break                                # kein erlaubter Schritt oder kein Fortschritt: steckt fest
        plan.append(schritt)
        lage = neu
    return plan, lage


def teste(namen, naechster_schritt):
    for name in namen:
        n, lage = lies_stadt('drohnen_eingabe_' + name + '.txt')
        plan, ende = finde_plan(n, lage, naechster_schritt)
        falsch = 0
        for haus in ende:
            if ende[haus] != haus:
                falsch = falsch + 1
        if falsch == 0:
            print(f'{name}: Schranke {untere_schranke(lage)}, fertig nach {len(plan)} Schritten mit {anzahl_wuerfe(plan)} Würfen, Schiedsrichter: {pruefe_plan(n, lage, plan)}')
        else:
            print(f'{name}: Schranke {untere_schranke(lage)}, steckt nach {len(plan)} Schritten fest, {falsch} Pakete liegen falsch, größter Abstand {untere_schranke(ende)}')

teste(['beispiel', 'stadt5', 'amacity'], schritt_tauschen)
```

Das Verfahren ist beim kleinen Beispiel **optimal**: 4 Schritte wie die Schranke. Bei `stadt5` und `amacity` **steckt es fest**: Es gibt keinen Tausch mehr, der die Kosten senkt, obwohl noch Pakete falsch liegen.
Das ist ein typischer Fall in der 2. Runde: Ein einfaches gieriges Verfahren erreicht nicht immer das Ziel. Dass es hier **ohne Schiedsrichter** nicht sofort aufgefallen wäre (die Zahl der Schritte sieht ja
unauffällig aus), zeigt, wie wichtig die Kontrolle ist. Dass die Pläne bis zum Stillstand **erlaubt** sind, sagt der Schiedsrichter.

</details>

### Besser: ein ganzer Schritt als Zuordnung

Statt einzelner Tausche suchen wir den **besten ganzen Schritt auf einmal**. In einem Schritt gibt **jedes** Haus sein Paket an **ein** Haus weiter (an sich selbst oder einen Nachbarn),
und **jedes** Haus bekommt genau **ein** Paket. Das ist ein **Zuordnungsproblem** (ähnlich wie in [Notebook 14](https://ktheu.github.io/bwinfN/notebooks/index.html?path=learn_bwinf1/14_zuordnen.ipynb) von learn_bwinf1): Zeilen der Tabelle sind die Häuser, von denen ein Paket kommt,
Spalten die Häuser, auf denen es landet. In die Tabelle kommen die Kosten des Pakets am neuen Ort, und alles, was nicht erlaubt ist (weiter als ein Nachbarhaus), bekommt riesige Kosten.
`scipy.optimize.linear_sum_assignment` löst das Problem mit der **Ungarischen Methode**, die du nicht selbst schreiben musst. Sie gibt zwei Listen zurück: Zeile `zeilen[k]` wird Spalte `spalten[k]` zugeordnet.

In [ ]:
RICHTUNG = {'N': (-1, 0), 'S': (1, 0), 'O': (0, 1), 'W': (0, -1)}


def lies_stadt(dateiname):
    """Gibt n und das Dictionary lage zurück: lage[(i, j)] = (a, b) heißt: auf Haus (i, j) liegt das Paket für Haus (a, b)."""
    with open('daten/' + dateiname, encoding='utf-8') as datei:
        zeilen = datei.read().split('\n')
    n = int(zeilen[0])
    lage = {}
    for zeile in zeilen[1:]:
        teile = zeile.split()
        if len(teile) == 4:
            i, j, a, b = [int(x) for x in teile]
            lage[(i, j)] = (a, b)
    return n, lage


def lies_plan(dateiname):
    """Ein Plan ist eine Liste von Schritten, ein Schritt ein Dictionary {Haus: Richtung} der werfenden Häuser."""
    zeilen = {}
    with open('daten/' + dateiname, encoding='utf-8') as datei:
        for zeile in datei.read().split('\n'):
            teile = zeile.split()
            if len(teile) == 3:
                zeilen[(int(teile[0]), int(teile[1]))] = teile[2]
    plan = []
    for k in range(len(zeilen[(0, 0)])):
        schritt = {}
        for haus in zeilen:
            if zeilen[haus][k] != '_':
                schritt[haus] = zeilen[haus][k]
        plan.append(schritt)
    return plan

def fuehre_schritt_aus(n, lage, schritt):
    """Gibt die neue Lage zurück oder einen Text, wenn der Schritt nicht erlaubt ist."""
    neu = {}
    for haus in lage:
        if haus in schritt:
            dz, ds = RICHTUNG[schritt[haus]]
            wohin = (haus[0] + dz, haus[1] + ds)
            if not (0 <= wohin[0] < n and 0 <= wohin[1] < n):
                return f'Haus {haus} wirft aus der Siedlung hinaus'
        else:
            wohin = haus                         # wer nicht wirft, behält sein Paket
        if wohin in neu:
            return f'auf Haus {wohin} landen zwei Pakete'
        neu[wohin] = lage[haus]
    return neu

def pruefe_plan(n, lage, plan):
    for k in range(len(plan)):
        lage = fuehre_schritt_aus(n, lage, plan[k])
        if isinstance(lage, str):                # ein Text: der Schritt war nicht erlaubt
            return f'Fehler in Schritt {k + 1}: {lage}'
    falsch = 0
    for haus in lage:
        if lage[haus] != haus:
            falsch = falsch + 1
    if falsch > 0:
        return f'am Ende liegen {falsch} Pakete falsch'
    return 'ok'


def anzahl_wuerfe(plan):
    summe = 0
    for schritt in plan:
        summe = summe + len(schritt)
    return summe

def abstand(haus, ziel):
    return abs(haus[0] - ziel[0]) + abs(haus[1] - ziel[1])


def untere_schranke(lage):
    """Mindestzahl der Schritte: der größte Abstand eines Pakets zu seinem Ziel."""
    groesster = 0
    for haus in lage:
        groesster = max(groesster, abstand(haus, lage[haus]))
    return groesster

def kosten(d):
    return d ** 3                                # weit entfernte Pakete zählen viel mehr (Vorrang)


def paketkosten(haus, ziel):
    return kosten(abstand(haus, ziel))


def gesamtkosten(lage):
    summe = 0
    for haus in lage:
        summe = summe + paketkosten(haus, lage[haus])
    return summe


def richtung(von, nach):
    for buchstabe in RICHTUNG:
        dz, ds = RICHTUNG[buchstabe]
        if (von[0] + dz, von[1] + ds) == nach:
            return buchstabe


def schritt_tauschen(n, lage):
    """Ein Schritt aus gleichzeitigen Tauschen: Nachbarn, die ihre Pakete tauschen, wenn die Kosten sinken."""
    kandidaten = []
    for a in lage:
        for b in [(a[0], a[1] + 1), (a[0] + 1, a[1])]:       # rechter und unterer Nachbar: jedes Paar nur einmal
            if b in lage:
                vorher = paketkosten(a, lage[a]) + paketkosten(b, lage[b])
                nachher = paketkosten(b, lage[a]) + paketkosten(a, lage[b])
                if nachher < vorher:
                    kandidaten.append((vorher - nachher, a, b))
    schritt = {}
    for gewinn, a, b in sorted(kandidaten, reverse=True):   # größter Gewinn zuerst
        if a not in schritt and b not in schritt:           # beide Häuser noch frei
            schritt[a] = richtung(a, b)
            schritt[b] = richtung(b, a)
    return schritt

def finde_plan(n, lage, naechster_schritt):
    """Wendet das übergebene Verfahren an, bis alles am Ziel ist oder die Kosten nicht mehr sinken."""
    plan = []
    while untere_schranke(lage) > 0 and len(plan) < 200:
        schritt = naechster_schritt(n, lage)
        neu = fuehre_schritt_aus(n, lage, schritt)
        if isinstance(neu, str) or gesamtkosten(neu) >= gesamtkosten(lage):
            break                                # kein erlaubter Schritt oder kein Fortschritt: steckt fest
        plan.append(schritt)
        lage = neu
    return plan, lage


def teste(namen, naechster_schritt):
    for name in namen:
        n, lage = lies_stadt('drohnen_eingabe_' + name + '.txt')
        plan, ende = finde_plan(n, lage, naechster_schritt)
        falsch = 0
        for haus in ende:
            if ende[haus] != haus:
                falsch = falsch + 1
        if falsch == 0:
            print(f'{name}: Schranke {untere_schranke(lage)}, fertig nach {len(plan)} Schritten mit {anzahl_wuerfe(plan)} Würfen, Schiedsrichter: {pruefe_plan(n, lage, plan)}')
        else:
            print(f'{name}: Schranke {untere_schranke(lage)}, steckt nach {len(plan)} Schritten fest, {falsch} Pakete liegen falsch, größter Abstand {untere_schranke(ende)}')

import numpy as np
from scipy.optimize import linear_sum_assignment


def nachbarn(n, haus):
    liste = []
    for dz, ds in [(-1, 0), (1, 0), (0, 1), (0, -1)]:
        if 0 <= haus[0] + dz < n and 0 <= haus[1] + ds < n:
            liste.append((haus[0] + dz, haus[1] + ds))
    return liste


def schritt_zuordnung(n, lage):
    haeuser = sorted(lage)
    nummer = {}
    for k in range(len(haeuser)):
        nummer[haeuser[k]] = k
    tabelle = np.full((len(haeuser), len(haeuser)), 10 ** 9)       # 10^9: verboten
    for von in haeuser:
        for nach in [von] + nachbarn(n, von):                      # behalten oder zu einem Nachbarn werfen
            tabelle[nummer[von], nummer[nach]] = paketkosten(nach, lage[von])
    zeilen, spalten = linear_sum_assignment(tabelle)
    schritt = {}
    for k in range(len(zeilen)):
        von = haeuser[zeilen[k]]
        nach = haeuser[spalten[k]]
        if von != nach:
            schritt[von] = richtung(von, nach)
    return schritt

teste(['beispiel', 'stadt5', 'amacity'], schritt_zuordnung)

beispiel: Schranke 4, fertig nach 4 Schritten mit 22 Würfen, Schiedsrichter: ok
stadt5: Schranke 7, fertig nach 8 Schritten mit 126 Würfen, Schiedsrichter: ok
amacity: Schranke 15, fertig nach 15 Schritten mit 1004 Würfen, Schiedsrichter: ok


Jetzt kommen alle drei Siedlungen ans Ziel, und der Schiedsrichter bestätigt jeden Plan:

| Siedlung | Schranke | gieriges Tauschen | Zuordnung je Schritt |
|---|---|---|---|
| `beispiel` (3 × 3) | 4 | 4 Schritte | 4 Schritte |
| `stadt5` (5 × 5) | 7 | steckt fest | 8 Schritte |
| `amacity` (10 × 10) | 15 | steckt fest | 15 Schritte |

Für `amacity` ist der Plan **bewiesen optimal**: 15 Schritte, und weniger als 15 gibt es wegen der Schranke nicht (im Lösungsnotebook steht derselbe Plan, mit Bildern).
Für `stadt5` hat der Plan 8 Schritte bei Schranke 7: **höchstens 14,3 % über dem Optimum** (1 / 7). Ob 7 Schritte möglich sind, wissen wir nicht, die Schranke muss nicht erreichbar sein.
Im Lösungsnotebook fragt MiniZinc bei kleinen Zufallsstädten, ob es einen Plan mit so vielen Schritten wie die Schranke gibt.

#### Frage E1.5

Du schreibst in die Dokumentation zu `stadt5`: „Der Plan hat 8 Schritte.“ Ergänze den Satz so, dass ein Leser **sieht**, wie gut das ist, und dass nichts Unbewiesenes behauptet wird.

✏️ *Deine Antwort (Doppelklick zum Bearbeiten):*

<details>
<summary>Lösung anzeigen (erst selbst überlegen!)</summary>

„Der Plan hat 8 Schritte, geprüft mit einem unabhängigen Schiedsrichter (alle Regeln erfüllt, am Ende liegt jedes Paket richtig). Mindestens nötig sind 7 Schritte, weil das am weitesten entfernte Paket 7 Häuser
von seinem Ziel liegt (Manhattan-Schranke). Der Plan liegt also **höchstens 14,3 % über dem Optimum**. Ob ein Plan mit 7 Schritten existiert, wissen wir nicht.“
Das sind vier Dinge: **was** herauskam, **dass** geprüft wurde, **wie** die Schranke begründet ist und **was offen** bleibt.

</details>

## Wettbewerbs-Ecke 2 (für Schnelle): Eine Mauer ohne übereinanderliegende Fugen

*Angelehnt an Aufgabe 1 „Die Kunst der Fuge“ der 2. Runde des 36. Bundeswettbewerbs Informatik (2017/18).*

Eine Mauer besteht aus Reihen. **Jede Reihe** besteht aus den `n` Steinen der Längen 1 bis `n`, in beliebiger Reihenfolge und lückenlos hintereinander.
Die Stellen zwischen den Steinen heißen **Fugen**. In keiner Stelle dürfen in **zwei Reihen** Fugen liegen (auch nicht in Reihen, die nicht direkt übereinanderliegen). Die Mauer soll **möglichst viele Reihen** haben.
Eine Mauer steht als Textdatei: eine Reihe pro Zeile, die Steinlängen durch Leerzeichen getrennt. `daten/mauer_n4.txt` ist eine Mauer für `n = 4` mit 3 Reihen, `mauer_n4_falsch.txt` hat einen Fehler.

#### Aufgabe E2.1

Schreibe `pruefe_mauer(mauer, n)`. Sie gibt `'ok'` zurück oder meldet den **ersten** Fehler: Eine Reihe enthält nicht genau die Längen 1 bis `n`, oder eine Fuge liegt an einer Stelle,
an der in einer **früheren** Reihe schon eine Fuge liegt (nenne beide Reihen). `fugen(reihe)` ist schon da.

**Erwartete Ausgabe:**
```
ok
Fuge bei 3 in Reihe 2 und in Reihe 3
Reihe 1 enthält nicht genau die Längen 1 bis 4
```

In [ ]:
import itertools


def lies_mauer(dateiname):
    mauer = []
    with open('daten/' + dateiname, encoding='utf-8') as datei:
        for zeile in datei.read().split('\n'):
            if zeile.strip() != '':
                mauer.append([int(x) for x in zeile.split()])
    return mauer


def fugen(reihe):
    """Stellen der Fugen einer Reihe: die Teilsummen ohne die letzte (das Ende der Mauer)."""
    lagen = []
    summe = 0
    for stein in reihe[:-1]:
        summe = summe + stein
        lagen.append(summe)
    return lagen

def pruefe_mauer(mauer, n):
    reihe_von = {}                               # Stelle -> Nummer der Reihe, in der dort schon eine Fuge liegt
    # dein Code
    return 'ok'


print(pruefe_mauer(lies_mauer('mauer_n4.txt'), 4))
print(pruefe_mauer(lies_mauer('mauer_n4_falsch.txt'), 4))
print(pruefe_mauer([[1, 2, 3, 3]], 4))

<details>
<summary>Tipp anzeigen</summary>

Für jede Reihe `nr`: erst `sorted(mauer[nr]) != list(range(1, n + 1))` prüfen, dann für jede Fuge `f` in `fugen(mauer[nr])`: Steht `f` schon in `reihe_von`, ist das der Fehler, sonst `reihe_von[f] = nr`.
(Reihen werden ab 1 gezählt, `nr` ab 0.)

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
import itertools


def lies_mauer(dateiname):
    mauer = []
    with open('daten/' + dateiname, encoding='utf-8') as datei:
        for zeile in datei.read().split('\n'):
            if zeile.strip() != '':
                mauer.append([int(x) for x in zeile.split()])
    return mauer


def fugen(reihe):
    """Stellen der Fugen einer Reihe: die Teilsummen ohne die letzte (das Ende der Mauer)."""
    lagen = []
    summe = 0
    for stein in reihe[:-1]:
        summe = summe + stein
        lagen.append(summe)
    return lagen

def pruefe_mauer(mauer, n):
    """Gibt 'ok' zurück oder beschreibt den ersten Fehler."""
    reihe_von = {}                               # Stelle -> Nummer der Reihe, in der dort schon eine Fuge liegt
    for nr in range(len(mauer)):
        if sorted(mauer[nr]) != list(range(1, n + 1)):
            return f'Reihe {nr + 1} enthält nicht genau die Längen 1 bis {n}'
        for f in fugen(mauer[nr]):
            if f in reihe_von:
                return f'Fuge bei {f} in Reihe {reihe_von[f] + 1} und in Reihe {nr + 1}'
            reihe_von[f] = nr
    return 'ok'

print(pruefe_mauer(lies_mauer('mauer_n4.txt'), 4))
print(pruefe_mauer(lies_mauer('mauer_n4_falsch.txt'), 4))
print(pruefe_mauer([[1, 2, 3, 3]], 4))
```

Die falsche Mauer hat in Reihe 2 und Reihe 3 beide eine Fuge an Stelle 3 (`2 + 1` und `3`). Die Meldung nennt Stelle und beide Reihen.

</details>

In [ ]:
def hoechstens(n):
    """Obere Schranke für die Zahl der Reihen (n >= 2)."""
    return n // 2 + 1

print(' n   Länge  Stellen  Fugen je Reihe  höchstens Reihen')
for n in range(2, 11):
    laenge = n * (n + 1) // 2
    print(f'{n:2} {laenge:6} {laenge - 1:7} {n - 1:14} {hoechstens(n):16}')

 n   Länge  Stellen  Fugen je Reihe  höchstens Reihen
 2      3       2              1                2
 3      6       5              2                2
 4     10       9              3                3
 5     15      14              4                3
 6     21      20              5                4
 7     28      27              6                4
 8     36      35              7                5
 9     45      44              8                5
10     55      54              9                6


**Wie hoch kann eine Mauer höchstens werden?** Jede Reihe ist `L = 1 + 2 + … + n = n·(n+1)/2` lang. Fugen können nur an den `L − 1` Stellen 1, 2, …, `L − 1` liegen (gemessen vom linken Rand),
und jede Reihe hat `n − 1` Fugen. Keine Stelle darf in zwei Reihen vorkommen. Verteilt man die `L − 1` Stellen auf Reihen mit je `n − 1` Fugen, passen höchstens `(L − 1) / (n − 1)` Reihen hinein
(**Schubfachprinzip**, wie in [Notebook 18](https://ktheu.github.io/bwinfN/notebooks/index.html?path=learn_bwinf1/18_mathematik.ipynb) von learn_bwinf1: Wären es mehr Reihen, müssten zwei Fugen auf einer Stelle liegen).
Mit `L − 1 = (n − 1)(n + 2) / 2` ist das genau `n/2 + 1`, bei ungeradem `n` abgerundet. In Python ist beides `n // 2 + 1`.

#### Frage E2.2

1. Rechne `(L − 1) / (n − 1)` für `n = 4` und für `n = 10` aus (`L − 1` ist die Zahl der Stellen).
2. Die Datei `mauer_n4.txt` hat 3 Reihen. Was sagt das über die Schranke für `n = 4`?
3. Was folgt aus der Schranke **nicht**?

✏️ *Deine Antwort (Doppelklick zum Bearbeiten):*

<details>
<summary>Lösung anzeigen (erst selbst überlegen!)</summary>

1. `n = 4`: `L = 10`, 9 Stellen, 3 Fugen je Reihe, also höchstens `9 / 3 = 3` Reihen. `n = 10`: `L = 55`, 54 Stellen, 9 Fugen je Reihe, also höchstens `54 / 9 = 6` Reihen.
2. Die Mauer hat 3 Reihen, mehr als 3 sind nicht möglich: Sie ist also **bewiesen optimal**. Alle 9 Stellen tragen genau eine Fuge.
3. Dass es für **jedes** `n` eine Mauer mit so vielen Reihen gibt. Die Schranke ist nur eine **obere** Grenze, das Programm muss zeigen, dass man sie erreicht (oder nicht).

</details>

#### Aufgabe E2.3

Das einfachste Verfahren baut die Mauer **Reihe für Reihe**: Für die nächste Reihe nimmt man die erste Anordnung der Steine, die keine schon belegte Stelle benutzt. `itertools.permutations(range(n, 0, -1))`
liefert alle Anordnungen der Längen `n` bis 1 (die mit langen Steinen links zuerst). Wird keine Anordnung mehr gefunden, ist die Mauer fertig.
Schreibe `reihe_fuer_reihe(n)`. Die Tabelle prüft jede Mauer mit `pruefe_mauer` und gibt den **Abstand zur Schranke** an (hier ist die Schranke eine **obere**: Abstand = fehlende Reihen in Prozent der Schranke).

**Erwartete Ausgabe:**
```
 n  Reihen  höchstens  Abstand
 2      2          2     0.0 %
 3      2          2     0.0 %
 4      2          3    33.3 %
 5      3          3     0.0 %
 6      3          4    25.0 %
 7      3          4    25.0 %
 8      3          5    40.0 %
```

In [ ]:
import itertools


def lies_mauer(dateiname):
    mauer = []
    with open('daten/' + dateiname, encoding='utf-8') as datei:
        for zeile in datei.read().split('\n'):
            if zeile.strip() != '':
                mauer.append([int(x) for x in zeile.split()])
    return mauer


def fugen(reihe):
    """Stellen der Fugen einer Reihe: die Teilsummen ohne die letzte (das Ende der Mauer)."""
    lagen = []
    summe = 0
    for stein in reihe[:-1]:
        summe = summe + stein
        lagen.append(summe)
    return lagen

def pruefe_mauer(mauer, n):
    """Gibt 'ok' zurück oder beschreibt den ersten Fehler."""
    reihe_von = {}                               # Stelle -> Nummer der Reihe, in der dort schon eine Fuge liegt
    for nr in range(len(mauer)):
        if sorted(mauer[nr]) != list(range(1, n + 1)):
            return f'Reihe {nr + 1} enthält nicht genau die Längen 1 bis {n}'
        for f in fugen(mauer[nr]):
            if f in reihe_von:
                return f'Fuge bei {f} in Reihe {reihe_von[f] + 1} und in Reihe {nr + 1}'
            reihe_von[f] = nr
    return 'ok'

def hoechstens(n):
    """Obere Schranke für die Zahl der Reihen (n >= 2)."""
    return n // 2 + 1

def reihe_fuer_reihe(n):
    belegt = set()                               # die Stellen, an denen schon eine Fuge liegt
    mauer = []
    # dein Code
    return mauer


print(' n  Reihen  höchstens  Abstand')
for n in range(2, 9):
    mauer = reihe_fuer_reihe(n)
    assert pruefe_mauer(mauer, n) == 'ok'
    s = hoechstens(n)
    print(f'{n:2} {len(mauer):6} {s:10} {100 * (s - len(mauer)) / s:7.1f} %')

<details>
<summary>Tipp anzeigen</summary>

`while True:`, darin `gefunden = None` und `for reihe in itertools.permutations(range(n, 0, -1)):`. Passt die Reihe (`not any(f in belegt for f in fugen(reihe))`), merke sie und mit `break` raus.
Ist `gefunden` danach noch `None`, `return mauer`, sonst die Reihe anhängen und ihre Fugen zu `belegt` hinzufügen.

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
import itertools


def lies_mauer(dateiname):
    mauer = []
    with open('daten/' + dateiname, encoding='utf-8') as datei:
        for zeile in datei.read().split('\n'):
            if zeile.strip() != '':
                mauer.append([int(x) for x in zeile.split()])
    return mauer


def fugen(reihe):
    """Stellen der Fugen einer Reihe: die Teilsummen ohne die letzte (das Ende der Mauer)."""
    lagen = []
    summe = 0
    for stein in reihe[:-1]:
        summe = summe + stein
        lagen.append(summe)
    return lagen

def pruefe_mauer(mauer, n):
    """Gibt 'ok' zurück oder beschreibt den ersten Fehler."""
    reihe_von = {}                               # Stelle -> Nummer der Reihe, in der dort schon eine Fuge liegt
    for nr in range(len(mauer)):
        if sorted(mauer[nr]) != list(range(1, n + 1)):
            return f'Reihe {nr + 1} enthält nicht genau die Längen 1 bis {n}'
        for f in fugen(mauer[nr]):
            if f in reihe_von:
                return f'Fuge bei {f} in Reihe {reihe_von[f] + 1} und in Reihe {nr + 1}'
            reihe_von[f] = nr
    return 'ok'

def hoechstens(n):
    """Obere Schranke für die Zahl der Reihen (n >= 2)."""
    return n // 2 + 1

def reihe_fuer_reihe(n):
    """Baut die Mauer Reihe für Reihe: je die erste Reihe, die keine belegte Stelle benutzt (lange Steine zuerst)."""
    belegt = set()
    mauer = []
    while True:
        gefunden = None
        for reihe in itertools.permutations(range(n, 0, -1)):
            if not any(f in belegt for f in fugen(reihe)):
                gefunden = list(reihe)
                break
        if gefunden is None:
            return mauer
        mauer.append(gefunden)
        belegt.update(fugen(gefunden))

print(' n  Reihen  höchstens  Abstand')
for n in range(2, 9):
    mauer = reihe_fuer_reihe(n)
    assert pruefe_mauer(mauer, n) == 'ok'
    s = hoechstens(n)
    print(f'{n:2} {len(mauer):6} {s:10} {100 * (s - len(mauer)) / s:7.1f} %')
```

Nur für `n = 2, 3` und `5` erreicht das Verfahren die Schranke. Bei `n = 4` baut es nur 2 Reihen, obwohl `mauer_n4.txt` 3 Reihen hat, und bei `n = 8` nur 3 statt 5 (40 % zu wenig).
Die Reihen werden **nacheinander** gebaut, und die ersten verteilen ihre Fugen ungeschickt, für spätere bleibt kein Platz.

</details>

Wie weit ist das Verfahren vom **Optimum** entfernt? Für kleine `n` probieren wir wie in Abschnitt 3 **alle** Mauern durch (alle Reihen als Anordnungen, jede Mauer nur einmal, indem die Reihen in aufsteigender Reihenfolge gewählt werden).
Für `n = 6` dauert es schon viel zu lange (720 Reihen), deshalb nur bis `n = 5`:

In [ ]:
import itertools


def lies_mauer(dateiname):
    mauer = []
    with open('daten/' + dateiname, encoding='utf-8') as datei:
        for zeile in datei.read().split('\n'):
            if zeile.strip() != '':
                mauer.append([int(x) for x in zeile.split()])
    return mauer


def fugen(reihe):
    """Stellen der Fugen einer Reihe: die Teilsummen ohne die letzte (das Ende der Mauer)."""
    lagen = []
    summe = 0
    for stein in reihe[:-1]:
        summe = summe + stein
        lagen.append(summe)
    return lagen

def pruefe_mauer(mauer, n):
    """Gibt 'ok' zurück oder beschreibt den ersten Fehler."""
    reihe_von = {}                               # Stelle -> Nummer der Reihe, in der dort schon eine Fuge liegt
    for nr in range(len(mauer)):
        if sorted(mauer[nr]) != list(range(1, n + 1)):
            return f'Reihe {nr + 1} enthält nicht genau die Längen 1 bis {n}'
        for f in fugen(mauer[nr]):
            if f in reihe_von:
                return f'Fuge bei {f} in Reihe {reihe_von[f] + 1} und in Reihe {nr + 1}'
            reihe_von[f] = nr
    return 'ok'

def hoechstens(n):
    """Obere Schranke für die Zahl der Reihen (n >= 2)."""
    return n // 2 + 1

def hoechste_mauer(n):
    """Größte mögliche Zahl der Reihen, durch Durchprobieren (nur für kleine n)."""
    reihen = []
    for reihe in itertools.permutations(range(1, n + 1)):
        reihen.append((list(reihe), set(fugen(reihe))))
    beste = {'hoehe': 0, 'mauer': []}

    def probiere(start, belegt, mauer):
        if len(mauer) > beste['hoehe']:
            beste['hoehe'] = len(mauer)
            beste['mauer'] = list(mauer)
        for k in range(start, len(reihen)):          # Reihen in aufsteigender Nummer: keine doppelten Mauern
            if not (reihen[k][1] & belegt):
                mauer.append(reihen[k][0])
                probiere(k + 1, belegt | reihen[k][1], mauer)
                mauer.pop()

    probiere(0, set(), [])
    return beste['hoehe'], beste['mauer']

for n in range(2, 6):
    hoehe, mauer = hoechste_mauer(n)
    print(n, hoehe, hoechstens(n), pruefe_mauer(mauer, n))

2 2 2 ok
3 2 2 ok
4 3 3 ok
5 3 3 ok


Das Optimum **erreicht die Schranke** für `n = 2` bis `5`: 2, 2, 3 und 3 Reihen. Bei `n = 4` fehlt Reihe für Reihe also eine Reihe, **obwohl** es eine Mauer mit 3 Reihen gibt: Es liegt am **Verfahren**, nicht an der Aufgabe.
Für größere `n` kann man das nicht mehr durchprobieren. Hier hilft die Schranke: Das [Lösungsnotebook](https://ktheu.github.io/bwinfN/notebooks/index.html?path=bwinf2/36_DieKunstDerFuge/fuge_loesung.ipynb) baut die Mauer **Spalte für Spalte** (alle Reihen gleichzeitig von links nach rechts)
und erreicht die Schranke für gerade `n`, zum Beispiel 4 Reihen bei `n = 6`, 6 bei `n = 10` und 56 bei `n = 110`. Dort sind die Mauern **bewiesen optimal**.

#### Frage E2.4

Reihe für Reihe liefert bei `n = 8` nur 3 Reihen. Welche der folgenden Aussagen darfst du in der Dokumentation schreiben?

1. „Für `n = 8` gibt es höchstens 3 Reihen.“
2. „Mein Verfahren baut für `n = 8` 3 Reihen. Mehr als 5 sind nicht möglich.“
3. „Mein Verfahren erreicht 60 % der Schranke.“

✏️ *Deine Antwort (Doppelklick zum Bearbeiten):*

<details>
<summary>Lösung anzeigen (erst selbst überlegen!)</summary>

**2** (und **3**, wenn man dazusagt, was die Schranke ist) sind ehrlich. **1** ist falsch: Wir wissen nur, dass **dieses Verfahren** 3 Reihen schafft. Dass nicht mehr geht, haben wir nicht gezeigt, die Schranke sagt 5,
und bei `n = 4` und `n = 6` geht es mit mehr Reihen, als Reihe für Reihe findet. Eine **Heuristik** liefert eine **obere Schranke für das Optimum** (3 ≤ Optimum), die Schubfach-Rechnung eine **untere Grenze der Möglichkeiten** (Optimum ≤ 5).

</details>

## Zum Schluss: Das Behälterproblem mit MiniZinc

Für **mittelgroße** Fälle, bei denen Durchprobieren in Python zu langsam ist (Abschnitt 3), kann ein **Solver** das Optimum finden. Mit ihm lässt sich die Güte einer Heuristik auch dort messen.
Du kennst von den MiniZinc-Arbeitsblättern (`forall`, `sum`, Variablen als Index, `.dzn`). Das Modell steht unten. **MiniZinc läuft nicht in JupyterLite:** Kopiere es in die MiniZinc IDE
oder in den [MiniZinc Playground](https://play.minizinc.org/) und füge die Daten als `.dzn` dazu (die Python-Zelle darunter erzeugt sie).

```minizinc
int: n;                                   % Zahl der Pakete
int: K;                                   % Kapazität eines Wagens
int: m;                                   % so viele Wagen stehen höchstens zur Verfügung (aus First Fit Decreasing)
int: schranke;                            % untere Schranke aus Abschnitt 4
array[1..n] of int: g;                    % Gewichte der Pakete

array[1..n] of var 1..m: wagen;           % wagen[i]: Nummer des Wagens von Paket i (hier ab 1)

% Kein Wagen ist überladen. (wagen[i] == w) ist wahr oder falsch und zählt in der Summe als 1 oder 0.
constraint forall(w in 1..m)(sum(i in 1..n)(g[i] * (wagen[i] == w)) <= K);

% Symmetrie brechen: Die Wagen sind gleichwertig, also kommt Paket 1 auf Wagen 1, und jedes weitere Paket
% höchstens auf den ersten noch unbenutzten Wagen.
constraint wagen[1] = 1;
constraint forall(i in 2..n)(wagen[i] <= 1 + max(j in 1..i-1)(wagen[j]));

var schranke..m: anzahl = max(wagen);     % Zahl der benutzten Wagen, nie unter der Schranke
solve minimize anzahl;
output ["anzahl = \(anzahl)\n", "wagen = \(wagen)\n"];
```

Was hier über die Arbeitsblätter hinausgeht:

- `(wagen[i] == w)` ist ein **Wahrheitswert**, MiniZinc rechnet in einer Summe damit wie mit 0 und 1.
- **Symmetriebrechung:** Vertauscht man in einer Lösung Wagen 1 und 2, erhält man eine gleichwertige. Ohne Gegenmaßnahme sucht der Solver alle diese Vertauschungen durch (bei 8 Wagen 40 320 mal dieselbe Lösung).
  Die beiden letzten `constraint`-Zeilen erlauben pro Lösung nur **eine** Nummerierung (Pakete zählen die Wagen der Reihe nach auf). Das ist dieselbe Idee wie in `exakt` in Abschnitt 3 („ein neuer Wagen ist immer der nächste“).
- **Schranken als Hilfe:** `m` (Ergebnis von First Fit Decreasing) begrenzt die Wagenzahl nach oben, `schranke` aus Abschnitt 4 nach unten. Die Schranke ist **unser Wissen**, das wir dem Solver mitgeben.
  Dass der Solver sie selbst nicht kennt, werden wir sehen.

In [ ]:
K = 20                                       # Kapazität eines Wagens in kg
pakete = [7, 9, 10, 7, 4, 7, 8, 3, 4]        # Gewichte der Pakete in kg

def pruefe(pakete, K, zuordnung):
    """Gibt 'ok' zurück oder beschreibt den ersten Verstoß."""
    if len(zuordnung) != len(pakete):
        return f'{len(zuordnung)} Einträge für {len(pakete)} Pakete'
    last = {}                                # last[w]: Gewicht auf Wagen w
    for i in range(len(pakete)):
        w = zuordnung[i]
        if w < 0:
            return f'Paket {i} hat die ungültige Wagennummer {w}'
        last[w] = last.get(w, 0) + pakete[i]
    for w in sorted(last):
        if last[w] > K:
            return f'Wagen {w} ist überladen: {last[w]} kg bei höchstens {K} kg'
    return 'ok'


def wagenzahl(zuordnung):
    return len(set(zuordnung))

def first_fit(pakete, K, reihenfolge):
    """reihenfolge: Liste der Paketnummern in der Reihenfolge, in der sie verteilt werden."""
    frei = []                                # frei[w]: noch freie Kapazität von Wagen w
    zuordnung = [0] * len(pakete)
    for i in reihenfolge:
        gefunden = -1
        for w in range(len(frei)):
            if frei[w] >= pakete[i]:
                gefunden = w
                break                        # der erste passende Wagen
        if gefunden == -1:                   # keiner passt: neuer Wagen
            frei.append(K)
            gefunden = len(frei) - 1
        frei[gefunden] = frei[gefunden] - pakete[i]
        zuordnung[i] = gefunden
    return zuordnung

def ff_gegeben(pakete, K):
    return first_fit(pakete, K, list(range(len(pakete))))


def ffd(pakete, K):
    # Paketnummern nach Gewicht absteigend: key=lambda wie in Notebook 03 von learn_bwinf1
    reihenfolge = sorted(range(len(pakete)), key=lambda i: -pakete[i])
    return first_fit(pakete, K, reihenfolge)

def schranke_gewicht(pakete, K):
    return -(-sum(pakete) // K)              # aufgerundet: S / K


def schranke_gross(pakete, K):
    anzahl = 0
    for p in pakete:
        if 2 * p > K:                        # schwerer als K / 2
            anzahl = anzahl + 1
    return anzahl


def schranke(pakete, K):
    return max(schranke_gewicht(pakete, K), schranke_gross(pakete, K))

def als_dzn(pakete, K):
    m = wagenzahl(ffd(pakete, K))
    return f'n = {len(pakete)};\nK = {K};\nm = {m};\nschranke = {schranke(pakete, K)};\ng = {pakete};'


print(als_dzn(pakete, K))

n = 9;
K = 20;
m = 4;
schranke = 3;
g = [7, 9, 10, 7, 4, 7, 8, 3, 4];


**Probe an den Paketen aus Abschnitt 1** (`K = 20`). Hier ist First Fit Decreasing einen Wagen zu schlecht, das Optimum ist 3:

**Ausgabe:**

**Ausgabe:**

```
anzahl = 3
wagen = [1, 2, 3, 3, 1, 2, 1, 3, 2]
----------
==========
```

**Vergleich mit dem Durchprobieren:** Für die 20 kleinen Fälle `zufallsfall(startwert, 10, 3, 12)` mit Startwert 1 bis 20 und `K = 20` liefert MiniZinc dasselbe Optimum wie `exakt` in Python (geprüft beim Erzeugen dieses Notebooks).
So kann man dem Solver auch bei Fällen trauen, für die man ihn braucht, weil Durchprobieren nicht mehr geht.

**Mittelgroße Fälle:** 20 Pakete (Gewichte 10 bis 60, `K = 100`, `zufallsfall(startwert, 20, 10, 60)`). In Python braucht `exakt` hier schon beim Startwert 0 über 7 Millionen Aufrufe (zu langsam für den Browser).
Chuffed findet das Optimum und beweist es jeweils in unter einer Sekunde (gemessen beim Erzeugen):

| Startwert | First Fit | First Fit Decreasing | Schranke | Optimum (MiniZinc) |
|---|---|---|---|---|
| 0 | 8 | 8 | 7 | 7 |
| 1 | 8 | 8 | 8 | 8 |
| 2 | 8 | 8 | 7 | 7 |
| 3 | 8 | 8 | 8 | 8 |

Jetzt kennen wir die **echte Güte**: Bei Startwert 0 und 2 liegt First Fit Decreasing einen Wagen (14,3 %) über dem Optimum, bei den anderen beiden ist es optimal. **Die Schranke ist in allen vier Fällen genau das Optimum**,
sie war hier also ein ehrliches Maß. Wo sie das Optimum nicht erreicht, sieht man den Unterschied erst durch solch einen exakten Lauf.

**Grenzen des Solvers.** Mit 30 Paketen kommt es auf den Fall an: Bei den Startwerten 0 und 2 ist First Fit Decreasing mit 11 Wagen gleich der Schranke 11. Dann ist das Optimum auch **ohne** Solver bewiesen,
und der Solver bestätigt es sofort (in der Tabelle von Abschnitt 5 wären das Fälle „Schranke erreicht“). Bei den Startwerten 1 und 3 liegt First Fit Decreasing mit 12 Wagen **über** der Schranke 11.
Dort musste Chuffed beim Messen sehr lange suchen (mehr als eine halbe Minute), bei Startwert 1 fand er 12 Wagen, konnte aber auch nach 30 Sekunden **nicht beweisen**, dass 11 unmöglich ist.
Bei 40 Paketen ist es ähnlich: Startwert 2 (First Fit Decreasing 14, Schranke 13) blieb offen, die anderen drei Fälle waren sofort fertig. Weil sich die Rechenzeit **nicht vorhersagen** lässt,
setzt man dem Solver immer ein **Zeitlimit** und schreibt ehrlich in die Dokumentation, was er in der Zeit **bewiesen** hat und was **offen** blieb. Ohne die Schranke als Bedingung im Modell
(`var 1..m: anzahl` statt `var schranke..m: anzahl`) war es noch deutlich schlechter: Schon bei 30 Paketen wurde nirgends ein Beweis fertig.

## Das hast du gelernt

```python
# Schiedsrichter: unabhängig vom Verfahren, meldet den ersten Verstoß genau
def pruefe(eingabe, loesung):
    if loesung hat die falsche Form:
        return 'Text: was ist falsch'
    for jede Regel der Aufgabe:
        if Regel verletzt:
            return 'Text: welche Regel, wo (Nummer, Wert)'
    return 'ok'

# Funktionen als Werte: das Verfahren wird übergeben
def bewerte(verfahren, eingabe):
    loesung = verfahren(eingabe)
    urteil = pruefe(eingabe, loesung)
    if urteil != 'ok':
        raise ValueError('Schiedsrichter: ' + urteil)
    return Güte(loesung)

# Güte als Abstand zur unteren Schranke, in Prozent
prozent = 100 * (ergebnis - schranke) / schranke
```

- Jede Lösung geht **vor** jeder Auswertung durch einen **Schiedsrichter**, auch Lösungen aus Dateien (Datei neu einlesen). Ein **zu milder** Schiedsrichter ist gefährlicher als ein zu strenger.
- **Funktionen kann man wie Werte übergeben** (`bewerte(ffd, ...)`, `[('First Fit', ff_gegeben), ...]`), so vergleicht man Verfahren mit demselben Schiedsrichter.
- Eine **untere Schranke** braucht eine **Begründung** (hier: Gewicht, große Pakete, Manhattan-Abstand, Schubfachprinzip) und einen **Grenzfall-Test** (`>` oder `>=`?).
  Ist die Lösung gleich der Schranke, ist sie **bewiesen optimal**. Sonst liegt sie **höchstens** so viel über dem Optimum, wie sie über der Schranke liegt. Die Schranke ist nicht immer erreichbar.
- Für **kleine** Fälle berechnet man das Optimum durch Durchprobieren (mit Abschneiden und ohne Symmetrie), bei **mittelgroßen** hilft ein Solver wie MiniZinc, mit Zeitlimit und ehrlicher Angabe, was offen blieb.
- Die **Güte** steht als **Tabelle über viele Zufallsfälle** mit festem Startwert in der Dokumentation: wie oft die Schranke erreicht wird, im Mittel und im schlechtesten Fall wie viel Prozent darüber.
- Ehrlich formulieren: „In 200 Fällen **höchstens x % über der Schranke**, also höchstens x % über dem Optimum“, nicht „immer“ und nicht „nur x % neben dem Optimum“.

### Aufgaben der 2. Runde dazu

Bearbeite eine Aufgabe zuerst selbst und sieh dir das Lösungsnotebook erst danach an.

- [Missglückte Drohnenlieferung](https://ktheu.github.io/bwinfN/notebooks/index.html?path=bwinf2/34_MissglueckteDrohnenlieferung/drohnen_loesung.ipynb) (34. BwInf): Schiedsrichter für Pläne, Manhattan-Schranke, gieriges Tauschen und ein Schritt als Zuordnung; mit MiniZinc für kleine Siedlungen
- [Die Kunst der Fuge](https://ktheu.github.io/bwinfN/notebooks/index.html?path=bwinf2/36_DieKunstDerFuge/fuge_loesung.ipynb) (36.): Schranke für die Höhe aus dem Schubfachprinzip, Reihe für Reihe gegen Spalte für Spalte

Im nächsten Notebook (02) geht es um **Struktur erkennen**: Erst nachdenken, dann programmieren, mit erzwungenen Reihenfolgen, Formeln statt Suche und Invarianten, und mit Durchprobieren bei kleinen Fällen als Bestätigung.